# OBSonix, sonde anatomie identifiable : quatre catégories, sources tenues hors

**Question.** Les 95 % d'exactitude « par image » de la sonde anatomie (rapport D du 27 juillet) tombaient à 14 % « par source », mais l'analyse était confondue : cinq catégories absentes de l'entraînement formaient 68,8 % du test et dix catégories sur dix-sept ne venaient que d'une seule source. Ce notebook refait la mesure sur les seules catégories identifiables, celles présentes dans au moins trois sources avec au moins vingt images par cellule (verdict S15 du 1er août 2026 : fœtal, cardiaque, thyroïde, sein), avec des plis où chaque source est tenue hors exactement une fois et où chaque pli de test contient les quatre catégories.

**Ce qu'il produit.** Pour les huit encodeurs (aléatoire, ImageNet-1k, DINOv2, A0 ep100/200/300, fœtal ep50/100) : exactitude équilibrée, exactitude brute et F1 macro sous découpage par image et sous sources tenues hors, avec intervalles bootstrap (par image et par grappe de sources), test de permutation, lignes de base (classe majoritaire, majorité de la source), contrôle d'identification de la source (globale et à catégorie fixée), ablation par centrage par source, tables CSV, Markdown et LaTeX, figures PNG 300 dpi et SVG, un README avec le paragraphe de méthodes en anglais et un verdict pré-enregistré.

**Comment l'exécuter.** Runtime GPU (A100 ou G4 de préférence), puis « Exécution » → « Tout exécuter ». Le montage du Drive demande une autorisation la première fois ; ensuite tout est autonome. En cas de coupure, relancer « Tout exécuter » : chaque étape porte un marqueur et n'est jamais refaite ; les résultats intermédiaires sont relus depuis le Drive.

**Où ça écrit.** Uniquement dans `MyDrive/OBSonix/eval/analyses3_anatomie4/` (résultats) et `MyDrive/OBSonix/eval/etats3_anatomie4/` (marqueurs). Rien n'est écrit dans `analyses2`, `reports`, `etats2`, le corpus ou les checkpoints. Un changement de paramètre après le premier lancement est refusé tant que le nom de campagne n'est pas changé, pour que rien ne se mélange.

**Publication GitHub.** L'étape S9 pousse les résultats (tables, figures, métriques, README, journal) dans `experiments/EXP25_ANATOMY_PROBE_SOURCE_HELD_OUT/` du dépôt `Fetal-odyssey/ObSonix` et une copie du notebook dans `notebooks/`, en un seul commit ne contenant que ce qui a changé. Elle lit le jeton dans le secret Colab `GOLDBACH` : dans le panneau « Secrets » (icône clé), le secret doit exister et « Accès depuis le notebook » doit être activé pour ce notebook ; sinon Colab affiche une demande d'accès pendant l'exécution de S9, qui attend jusqu'à 90 s. Sans accord, l'étape est simplement ignorée et tout reste sur le Drive : activer l'accès puis relancer la seule cellule S9 (puis S10) suffit, la session restant ouverte. Le jeton n'est jamais affiché ni écrit. Lien pour relancer le notebook depuis GitHub : `https://colab.research.google.com/github/Fetal-odyssey/ObSonix/blob/main/notebooks/OBSonix_sonde_anatomie4.ipynb`.

**Étapes.** S0 environnement et pré-enregistrement · S1 plan d'échantillonnage et plis · S2 extraction des images depuis les shards · S3 features des huit encodeurs · S4 sondes anatomie · S5 contrôles source · S6 ablation par centrage · S7 tables, figures, README, manifeste · S8 dépouillement de la lecture en aveugle (seulement si la grille est complète) · S9 publication dans le dépôt GitHub · S10 état final.


## S0 : environnement, garde-fous, reprise, pré-enregistrement

Monte le Drive, installe les dépendances, clone le dépôt I-JEPA de référence (commit 52c1ae9) avec le patch FlashAttention, vérifie l index et les cinq checkpoints, crée le dossier de campagne et écrit le pré-enregistrement (question, critère principal, règle de décision) lors du premier lancement. Toute modification ultérieure des paramètres est refusée tant que `CAMPAGNE` garde le même nom.

In [ ]:
# ============================================================================
# S0 — ENVIRONNEMENT, GARDE-FOUS, REPRISE, PRÉ-ENREGISTREMENT
# ============================================================================
RACINE      = '/content/drive/MyDrive/OBSonix'
CAMPAGNE    = 'analyses3_anatomie4'   # dossier de sortie ; changer ce nom pour toute nouvelle configuration
GRAINE      = 1789
CATEGORIES  = ['foetal', 'cardiaque', 'thyroide', 'sein']   # pré-enregistrées : verdict S15 du 1er août 2026
MIN_CELLULE = 20      # images minimum par (catégorie, source) pour que la source soit retenue
PLAFOND     = 300     # images maximum tirées par (catégorie, source)
K_PLIS      = 4       # plis « sources tenues hors » ; retombe à 3 si aucune affectation ne satisfait la contrainte
N_BOOT      = 1000    # rééchantillonnages bootstrap
N_PERM      = 500     # permutations pour le niveau de hasard
FAIRE_S6    = True    # ablation par centrage par source
FORCE       = set()   # ex. {'S4', 'S7'} pour refaire une étape malgré son marqueur
# Jeux apparentés (même institution ou même campagne d acquisition) : tenus hors ENSEMBLE, jamais l un en train et l autre en test
FAMILLES    = {'EchoNet_Dynamic': 'EchoNet', 'EchoNet_LVH': 'EchoNet', 'EchoNet_Pediatric': 'EchoNet',
               'FE1_FetalEcho_T1': 'FetalEcho', 'FE2_FetalEcho_T2': 'FetalEcho'}

import os, sys, io, json, gc, time, csv, hashlib, tarfile, zipfile, shutil, subprocess, importlib, warnings, traceback
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

MOCK = os.environ.get('OBSONIX_MOCK') == '1'          # mode test hors Colab (jeu de données factice)
if MOCK:
    RACINE = os.environ['OBSONIX_MOCK_RACINE']
    PLAFOND = int(os.environ.get('OBSONIX_MOCK_PLAFOND', '15'))
    N_BOOT, N_PERM = 100, 50
    FAMILLES = {'fetalA': 'fetalAB', 'fetalB': 'fetalAB'}

if not MOCK:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'scikit-learn',
                    'pandas', 'pyarrow', 'matplotlib', 'pillow'], check=False)
importlib.invalidate_caches()
import numpy as np, pandas as pd, torch
from PIL import Image

D = Path(RACINE)
if not D.exists() and not MOCK:
    from google.colab import drive
    drive.mount('/content/drive')
assert D.exists(), f'Racine introuvable : {D}. Vérifier le montage du Drive.'

EV     = D / 'eval'
OUT    = EV / CAMPAGNE
ETATS  = EV / ('etats3_' + CAMPAGNE.replace('analyses3_', ''))
LOCAL  = Path(os.environ['OBSONIX_MOCK_LOCAL']) if MOCK else Path('/content/local_' + CAMPAGNE)
SHARDS = D / 'media_master_V6' / 'shards_v6'
IDX    = EV / 'index_corpus.csv'
CKA, CKF = D / 'checkpoints' / 'A0_vitb16_256px', D / 'checkpoints' / 'FOETAL_vitb16_256px'
GRILLE_ADJ = EV / 'analyses2' / 'adjudication' / 'grille_de_lecture.csv'
CLE_ADJ    = EV / 'analyses2' / 'adjudication' / 'CLE_NE_PAS_OUVRIR_AVANT_LECTURE.csv'
for p in (OUT, ETATS, LOCAL, OUT / 'S2_parts', OUT / 'features', OUT / 'metrics',
          OUT / 'tables', OUT / 'figures', OUT / 'archive', LOCAL / 'images'):
    p.mkdir(parents=True, exist_ok=True)
JR = OUT / 'JOURNAL.md'

# ------------------------------------------------------------------ journal
def jrn(m, bat_=False):
    t = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with JR.open('a', encoding='utf-8') as f:
        f.write(f'- `{t}` {"~ " if bat_ else ""}{m}\n')
    if not bat_:
        print(f'[{t}] {m}', flush=True)

class Progres:
    """Barre, débit et temps restant à l'écran ; battement de cœur au journal toutes les 20 s."""
    def __init__(self, total, nom, chaque=1):
        self.total, self.nom, self.n = max(int(total), 1), nom, 0
        self.t0 = self.tj = time.time(); self.chaque = max(1, int(chaque))
    @staticmethod
    def _fmt(s):
        s = max(int(s), 0); h, r = divmod(s, 3600); m, sec = divmod(r, 60)
        return f'{h}h{m:02d}m' if h else (f'{m}m{sec:02d}s' if m else f'{sec}s')
    def __call__(self, n=1):
        self.n += n
        if self.n % self.chaque and self.n < self.total:
            return
        e = time.time() - self.t0; deb = self.n / max(e, 1e-9)
        f = int(28 * min(self.n / self.total, 1))
        print(f'\r  {self.nom:26s} [{"█" * f}{"░" * (28 - f)}] {self.n:>6}/{self.total} '
              f'{100 * self.n / self.total:5.1f}% | {deb:7.1f}/s | reste '
              f'{self._fmt((self.total - self.n) / max(deb, 1e-9))}   ', end='', flush=True)
        if time.time() - self.tj > 20:
            self.tj = time.time(); jrn(f'{self.nom} {self.n}/{self.total}', True)
    def fin(self):
        print(f'\r  {self.nom:26s} terminé : {self.n} en {self._fmt(time.time() - self.t0)}' + ' ' * 40, flush=True)
        jrn(f'{self.nom} terminé : {self.n} en {self._fmt(time.time() - self.t0)}')

# ------------------------------------------------- marqueurs et écritures sûres
def fait(n):
    return (ETATS / f'{n}.ok').exists()

def a_faire(n):
    if n in FORCE and fait(n):
        (ETATS / f'{n}.ok').unlink(); jrn(f'{n} : marqueur retiré (FORCE)')
    return not fait(n)

def marque(n, resume=''):
    p = ETATS / f'{n}.ok'
    p.write_text(f'{datetime.now():%Y-%m-%d %H:%M:%S}\n{resume}\n', encoding='utf-8')
    jrn(f'{n} : TERMINÉ ET VÉRIFIÉ ; {resume}')

def _dans_perimetre(p):
    p = Path(p).resolve()
    return any(str(p).startswith(str(r.resolve())) for r in (OUT, ETATS, LOCAL))

def sha256_fichier(p, n=16):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for bloc in iter(lambda: f.read(1 << 20), b''):
            h.update(bloc)
    return h.hexdigest()[:n]

def ecrire_atomique(chemin, fn_ecriture):
    """GARDE-FOU : n'écrit que dans le périmètre de la campagne ; écriture temporaire puis renommage."""
    chemin = Path(chemin)
    assert _dans_perimetre(chemin), f'Écriture refusée hors périmètre : {chemin}'
    chemin.parent.mkdir(parents=True, exist_ok=True)
    tmp = chemin.with_name(chemin.name + '.tmp')
    fn_ecriture(tmp)
    tmp.replace(chemin)
    return chemin

def ecrire_df(df, nom, sous='metrics'):
    """CSV et Parquet dans OUT/<sous>/ ; l'ancienne version est archivée avant d'être remplacée."""
    for ext in ('csv', 'parquet'):
        p = OUT / sous / f'{nom}.{ext}'
        if p.exists():
            shutil.copy2(p, OUT / 'archive' / f'{nom}_{datetime.now():%Y%m%d_%H%M%S}.{ext}')
        if ext == 'csv':
            ecrire_atomique(p, lambda t: df.to_csv(t, index=False))
        else:
            try:
                ecrire_atomique(p, lambda t: df.to_parquet(t, index=False))
            except Exception as e:
                jrn(f'parquet non écrit pour {nom} ({type(e).__name__}) ; le CSV fait foi')
    print(f'  écrit : {sous}/{nom} ({len(df)} lignes)')

def lire_df(nom, sous='metrics'):
    p = OUT / sous / f'{nom}.csv'
    return pd.read_csv(p) if p.exists() else None

def ecrire_texte(chemin, texte):
    ecrire_atomique(chemin, lambda t: Path(t).write_text(texte, encoding='utf-8'))

def echec(etape, e):
    jrn(f'{etape} : ÉCHEC {type(e).__name__} : {e}')
    with JR.open('a', encoding='utf-8') as f:
        f.write('```\n' + traceback.format_exc() + '```\n')
    raise

# --------------------------------------------------- vérifications d'entrée
assert IDX.exists(), f'Index du corpus introuvable : {IDX}'
ENCODEURS = {
    'RANDOM': None, 'IN1K': 'timm', 'DINOv2': 'timm',
    'A0 ep100':     CKA / 'OBSonix_A0_vitb16_256px_ep100.pth.tar',
    'A0 ep200':     CKA / 'OBSonix_A0_vitb16_256px_ep200.pth.tar',
    'A0 ep300':     CKA / 'OBSonix_A0_vitb16_256px_ep300_FINAL.pth.tar',
    'FOETAL ep50':  CKF / 'OBSonix_FOETAL_vitb16_256px_ep50.pth.tar',
    'FOETAL ep100': CKF / 'OBSonix_FOETAL_vitb16_256px_ep100.pth.tar',
}
PRINCIPAL, TEMOIN = 'A0 ep300', 'DINOv2'
if MOCK:
    ENCODEURS = {'RANDOM': None, 'RANDOM_bis': None}
    PRINCIPAL, TEMOIN = 'RANDOM', 'RANDOM_bis'
else:
    manquants = [nm for nm, p in ENCODEURS.items() if isinstance(p, Path) and not p.exists()]
    assert not manquants, f'Checkpoints absents : {manquants}. Vérifier {CKA} et {CKF}.'
    for nm, p in ENCODEURS.items():
        if isinstance(p, Path):
            taille = p.stat().st_size / 1e9
            assert taille > 1.0, f'{nm} : fichier suspect ({taille:.2f} Go)'
    shards_presents = sorted(SHARDS.glob('obsonix-v6-*.tar'))
    assert len(shards_presents) == 32, f'{len(shards_presents)} shards trouvés au lieu de 32 dans {SHARDS}'

# --------------------------------------------------- dépôt I-JEPA et FlashAttention
IJEPA = Path(os.environ['OBSONIX_MOCK_IJEPA']) if MOCK else Path('/content/ijepa')
if not IJEPA.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/facebookresearch/ijepa', str(IJEPA)], check=True)
    subprocess.run(['git', '-C', str(IJEPA), 'checkout', '-q', '52c1ae9'], check=True)
os.environ['TORCHDYNAMO_DISABLE'] = '1'        # torch.compile incompatible avec ce dépôt
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
VT = IJEPA / 'src' / 'models' / 'vision_transformer.py'; srcvt = VT.read_text()
ANCIEN = '''        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x, attn'''
NOUVEAU = '''        x = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, dropout_p=(self.attn_drop.p if self.training else 0.0))
        x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x, None'''
if 'scaled_dot_product_attention' in srcvt:
    print('FlashAttention : déjà actif')
elif ANCIEN in srcvt:
    VT.write_text(srcvt.replace(ANCIEN, NOUVEAU, 1))
    for m in [k for k in list(sys.modules) if k.startswith('src.')]:
        del sys.modules[m]
    print('FlashAttention : patch SDPA appliqué')
else:
    print('/!\\ FlashAttention : motif introuvable, vérifier le commit (le calcul reste exact, seulement plus lent)')
if str(IJEPA) not in sys.path:
    sys.path.insert(0, str(IJEPA))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
BS_FEAT = 256 if VRAM > 60 else (64 if DEVICE == 'cuda' else 16)
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f'GPU : {torch.cuda.get_device_name(0) if VRAM else "absent (CPU, lent mais exact)"} | {VRAM:.1f} Go | lot features {BS_FEAT}')

# --------------------------------------------------- pré-enregistrement et verrou de configuration
CONFIG = dict(campagne=CAMPAGNE, graine=GRAINE, categories=CATEGORIES, min_cellule=MIN_CELLULE,
              plafond=PLAFOND, k_plis=K_PLIS, n_boot=N_BOOT, n_perm=N_PERM, familles=FAMILLES,
              encodeurs=list(ENCODEURS), principal=PRINCIPAL, temoin=TEMOIN,
              pretraitement='Resize((px,px)) + ToTensor, RGB, px=256 (I-JEPA, IN1K) ou 224 (DINOv2)',
              features='moyenne des tokens de patch (token de classe exclu), standardisés sur le train',
              sonde='régression logistique multinomiale (lbfgs, C=1, max_iter=3000), variantes équilibrée et brute')
p_cfg = OUT / 'S0_config.json'
if p_cfg.exists():
    ancienne = json.loads(p_cfg.read_text(encoding='utf-8'))
    cles_fixes = [k for k in CONFIG if k not in ('n_boot', 'n_perm')]
    diff = {k: (ancienne.get(k), CONFIG[k]) for k in cles_fixes if ancienne.get(k) != CONFIG[k]}
    assert not diff, ('CONFIGURATION DIFFÉRENTE de celle pré-enregistrée pour cette campagne : '
                      f'{diff}. Choisir un autre nom de CAMPAGNE plutôt que de mélanger les résultats.')
    print('Configuration identique à celle pré-enregistrée : reprise autorisée.')
else:
    ecrire_texte(p_cfg, json.dumps(CONFIG, ensure_ascii=False, indent=1))
    ecrire_texte(OUT / 'S0_preenregistrement.md', f'''# Pré-enregistrement, campagne {CAMPAGNE}

Date : {datetime.now():%Y-%m-%d %H:%M}

## Question
La représentation gelée des encodeurs sépare-t-elle l'anatomie (fœtal, cardiaque, thyroïde, sein) lorsque les sources
d'acquisition du test n'ont jamais été vues par la sonde ? L'analyse du 27 juillet (95 % par image, 14 % par source)
était confondue par des catégories absentes de l'entraînement ; celle-ci se restreint aux catégories présentes dans au
moins trois sources avec au moins {MIN_CELLULE} images par cellule, et chaque pli de test contient les quatre catégories.

## Critère principal
Exactitude équilibrée (moyenne des rappels par catégorie) de la sonde « sources tenues hors », prédictions hors pli
regroupées, avec intervalle de confiance bootstrap à 95 % par grappe de sources et test de permutation (N = {N_PERM}).
Règle de décision, fixée avant de voir les résultats : le signal anatomique survit à la tenue hors des sources pour un
encodeur si la borne inférieure de l'IC à 95 % par grappe dépasse le 97,5e centile de la distribution de permutation
et si l'exactitude équilibrée dépasse la ligne de base « classe majoritaire ». Le verdict global porte sur
{PRINCIPAL} (encodeur principal) et {TEMOIN} (témoin).

## Critères secondaires
Exactitude brute, F1 macro, rappel par catégorie, même mesure sous découpage par image (sources partagées),
identification de la source (globale et à catégorie fixée), ablation par centrage par source.

## Paramètres
{json.dumps(CONFIG, ensure_ascii=False, indent=1)}
''')
    print('Pré-enregistrement écrit :', OUT / 'S0_preenregistrement.md')

jrn(f'=== session démarrée ({"MOCK" if MOCK else "Colab"}) ; étapes faites : {[p.stem for p in sorted(ETATS.glob("*.ok"))]} ===')


## S1 : plan d échantillonnage et plis « sources tenues hors »

Recalcule l éligibilité des catégories depuis l index et la confronte à la liste pré-enregistrée, tire au plus `PLAFOND` images par cellule (catégorie, source), affecte chaque source à un pli de sorte que chaque pli de test contienne les quatre catégories et qu aucune source ne soit partagée entre entraînement et test, puis fixe un découpage par image (70/30 à l intérieur de chaque cellule) sur les mêmes images. Sorties : `metrics/S1_echantillon`, `metrics/S1_plan_plis`, `S1_resume.md`.

In [ ]:
# ============================================================================
# S1 — PLAN D'ÉCHANTILLONNAGE ET PLIS « SOURCES TENUES HORS »
# ============================================================================
def plan_plis(cellules, K, seed):
    """Affecte chaque source à un pli. Contrainte : chaque pli contient chaque catégorie.
    cellules : DataFrame (anatomie, source, n). Retourne (pli_par_source, effectifs, ok)."""
    rng = np.random.default_rng(seed)
    contrib = defaultdict(dict)
    for r in cellules.itertuples(index=False):
        contrib[r.source][r.anatomie] = int(r.n)
    pli = {}
    eff = {k: {c: 0 for c in CATEGORIES} for k in range(K)}
    ordre = cellules.groupby('anatomie').source.nunique().sort_values().index   # catégories rares d'abord
    for c in ordre:
        srcs = sorted(cellules[cellules.anatomie == c].source.unique())
        rng.shuffle(srcs)
        for s in srcs:
            if s in pli:
                continue
            k = min(range(K), key=lambda k: (eff[k][c], sum(eff[k].values()), rng.random()))
            pli[s] = k
            for cc, n in contrib[s].items():
                eff[k][cc] += n
    ok = all(eff[k][c] > 0 for k in range(K) for c in CATEGORIES)
    return pli, eff, ok

if a_faire('S1'):
    try:
        idx = pd.read_csv(IDX, dtype=str)
        idx['cle'] = idx['cle'].astype(str)
        jrn(f'S1 : index lu, {len(idx)} images, {idx.source.nunique()} sources, {idx.anatomie.nunique()} catégories')
        if not MOCK and len(idx) != 255887:
            jrn(f'S1 : AVERTISSEMENT index de {len(idx)} lignes au lieu des 255 887 attendues (corpus V6) ; on continue')

        # Éligibilité recalculée puis confrontée à la liste pré-enregistrée
        mat = pd.crosstab(idx.anatomie, idx.source)
        eligibles = sorted(a for a in mat.index if int((mat.loc[a] >= MIN_CELLULE).sum()) >= 3)
        jrn(f'S1 : catégories éligibles (≥ 3 sources à ≥ {MIN_CELLULE} images) : {eligibles}')
        assert set(CATEGORIES) <= set(eligibles), f'Catégories pré-enregistrées non éligibles : {set(CATEGORIES) - set(eligibles)}'
        if set(eligibles) - set(CATEGORIES):
            jrn(f'S1 : catégories éligibles non retenues (hors pré-enregistrement) : {sorted(set(eligibles) - set(CATEGORIES))}')

        # Cellules (catégorie, source) retenues
        sous = idx[idx.anatomie.isin(CATEGORIES)]
        cellules = sous.groupby(['anatomie', 'source']).size().reset_index(name='n')
        cellules = cellules[cellules.n >= MIN_CELLULE].reset_index(drop=True)
        for c in CATEGORIES:
            assert (cellules.anatomie == c).sum() >= 3, f'{c} : moins de 3 sources retenues'

        # Tirage plafonné par cellule, déterministe
        rng = np.random.default_rng(GRAINE)
        morceaux = []
        for r in cellules.itertuples(index=False):
            cell = sous[(sous.anatomie == r.anatomie) & (sous.source == r.source)].sort_values('cle')
            if len(cell) > PLAFOND:
                cell = cell.iloc[np.sort(rng.choice(len(cell), PLAFOND, replace=False))]
            morceaux.append(cell)
        ech = pd.concat(morceaux, ignore_index=True)[['cle', 'shard', 'source', 'anatomie']]
        assert ech.cle.is_unique, 'clés dupliquées dans l échantillon'
        ech['groupe'] = ech.source.map(lambda s_: FAMILLES.get(s_, s_))   # unité tenue hors : la famille de jeux

        # Plis : K_PLIS, sinon K_PLIS-1 ; graines successives jusqu'à satisfaire la contrainte
        cell_ech = ech.groupby(['anatomie', 'groupe']).size().reset_index(name='n').rename(columns={'groupe': 'source'})
        choisi = None
        for K in (K_PLIS, K_PLIS - 1):
            for seed in range(200):
                pli, eff, ok = plan_plis(cell_ech, K, GRAINE + seed)
                if ok:
                    choisi = (K, seed, pli, eff); break
            if choisi:
                break
        assert choisi, 'aucune affectation de sources aux plis ne satisfait la contrainte ; réduire K_PLIS'
        K, seed, pli, eff = choisi
        ech['pli'] = ech.groupe.map(pli).astype(int)
        jrn(f'S1 : {K} plis (graine {GRAINE + seed}) ; effectifs par pli et catégorie : {eff}')

        # Découpage « par image » : 70/30 à l'intérieur de chaque cellule (mêmes sources des deux côtés)
        ech['split_image'] = 'train'
        rng2 = np.random.default_rng(GRAINE + 1)
        for (a, s), g in ech.groupby(['anatomie', 'source']):
            n_te = max(1, int(round(0.3 * len(g))))
            te = rng2.choice(g.index.values, n_te, replace=False)
            ech.loc[te, 'split_image'] = 'test'

        # Garde-fous de conception
        for k in range(K):
            te_src = set(ech[ech.pli == k].groupe); tr_src = set(ech[ech.pli != k].groupe)
            assert not (te_src & tr_src), f'pli {k} : groupes de sources partagés entre train et test'
            assert set(ech[ech.pli == k].anatomie) == set(CATEGORIES), f'pli {k} : catégorie absente du test'
            assert set(ech[ech.pli != k].anatomie) == set(CATEGORIES), f'pli {k} : catégorie absente du train'
        for cle_ in ('HC18', 'PSFHS', 'IUGC', 'FH-PS-AOP', 'US_FETUS', 'AHU'):
            assert not ech.source.str.contains(cle_, case=False).any(), f'source d évaluation dans l échantillon : {cle_}'

        plan = pd.DataFrame([dict(source=s, groupe=FAMILLES.get(s, s), pli=int(pli[FAMILLES.get(s, s)]),
                                  categories=';'.join(sorted(ech[ech.source == s].anatomie.unique())),
                                  n_images=int((ech.source == s).sum()),
                                  n_images_corpus=int((idx.source == s).sum()))
                             for s in sorted(ech.source.unique(), key=lambda s_: (pli[FAMILLES.get(s_, s_)], s_))])
        ecrire_df(ech, 'S1_echantillon'); ecrire_df(plan, 'S1_plan_plis')
        effectifs = ech.pivot_table(index='pli', columns='anatomie', values='cle', aggfunc='count', fill_value=0)
        resume = ['# S1 : plan d échantillonnage', '',
                  f'Images tirées : {len(ech)} ; sources : {ech.source.nunique()} ; catégories : {CATEGORIES}',
                  f'Plafond par cellule : {PLAFOND} ; minimum par cellule : {MIN_CELLULE} ; plis : {K}', '',
                  'Effectifs par pli (test) et catégorie :', effectifs.to_markdown() if hasattr(effectifs, 'to_markdown') else effectifs.to_string(), '',
                  'Sources par pli :', plan.to_string(index=False)]
        ecrire_texte(OUT / 'S1_resume.md', '\n'.join(resume))
        # vérification par relecture avant marquage
        relu = lire_df('S1_echantillon')
        assert relu is not None and len(relu) == len(ech) and relu.cle.is_unique
        marque('S1', f'{len(ech)} images, {ech.source.nunique()} sources ({ech.groupe.nunique()} groupes), {K} plis')
    except Exception as e:
        echec('S1', e)
else:
    print('S1 : déjà fait, rechargement')

ech = lire_df('S1_echantillon'); ech['cle'] = ech['cle'].astype(str)
plan = lire_df('S1_plan_plis')
K = int(ech.pli.max()) + 1
print(ech.pivot_table(index='pli', columns='anatomie', values='cle', aggfunc='count', fill_value=0))
print(f'{len(ech)} images | {ech.source.nunique()} sources | {K} plis')


## S2 : extraction des images depuis les shards

Lit chaque shard une seule fois et range les octets d origine des images tirées dans une archive par shard (`S2_parts/`), vérifiée après écriture ; un shard déjà archivé n est jamais relu. Les images manquantes (moins de 1 %) sont listées et retirées. Une copie locale est faite pour les lectures rapides.

In [ ]:
# ============================================================================
# S2 — EXTRACTION DES IMAGES DEPUIS LES SHARDS (octets d'origine, sans ré-encodage)
# Une archive zip par shard dans OUT/S2_parts/ ; un shard terminé n'est jamais relu.
# ============================================================================
EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

def part_valide(p_zip, attendu):
    """Un zip de shard est valide s'il s'ouvre et contient exactement les clés attendues."""
    if not p_zip.exists():
        return False
    try:
        with zipfile.ZipFile(p_zip) as z:
            noms = {os.path.splitext(n)[0] for n in z.namelist()}
        return noms == set(attendu)
    except zipfile.BadZipFile:
        return False

if a_faire('S2'):
    try:
        par_shard = {sh: set(g.cle) for sh, g in ech.groupby('shard')}
        jrn(f'S2 : {len(ech)} images à extraire dans {len(par_shard)} shards')
        manquantes_total = []
        pr = Progres(len(par_shard), 'shards', chaque=1)
        for sh, cles in sorted(par_shard.items()):
            p_tar = SHARDS / sh if not str(sh).startswith('/') else Path(sh)
            if not p_tar.exists():
                p_tar = SHARDS / (str(sh) + '.tar')
            assert p_tar.exists(), f'shard introuvable : {sh}'
            p_zip = OUT / 'S2_parts' / (p_tar.stem + '.zip')
            p_manq = OUT / 'S2_parts' / (p_tar.stem + '.manquantes.txt')
            deja = set(p_manq.read_text().split()) if p_manq.exists() else set()
            if part_valide(p_zip, cles - deja):
                manquantes_total += sorted(deja); pr(); continue
            trouve = {}
            with tarfile.open(p_tar) as tar:
                for m in tar:
                    if not m.isfile():
                        continue
                    k, e = os.path.splitext(os.path.basename(m.name))
                    if e.lower() in EXTS and k in cles and k not in trouve:
                        trouve[k] = (e.lower(), tar.extractfile(m).read())
                        if len(trouve) == len(cles):
                            break
            manquantes = sorted(cles - set(trouve))
            def _ecrire_zip(t, trouve=trouve):
                with zipfile.ZipFile(t, 'w', compression=zipfile.ZIP_STORED) as z:
                    for k, (e, b) in sorted(trouve.items()):
                        z.writestr(k + e, b)
            ecrire_atomique(p_zip, _ecrire_zip)
            ecrire_texte(p_manq, '\n'.join(manquantes))
            assert part_valide(p_zip, set(trouve)), f'zip invalide après écriture : {p_zip.name}'
            manquantes_total += manquantes
            jrn(f'S2 : {p_tar.name} : {len(trouve)}/{len(cles)} images ({len(manquantes)} manquantes)')
            pr()
        pr.fin()
        part = len(manquantes_total) / len(ech)
        assert part < 0.01, f'{len(manquantes_total)} images introuvables ({100 * part:.2f} %) : vérifier l index et les shards'
        if manquantes_total:
            ecrire_texte(OUT / 'S2_manquantes.csv', 'cle\n' + '\n'.join(manquantes_total))
            jrn(f'S2 : {len(manquantes_total)} images introuvables, retirées de l échantillon (S2_manquantes.csv)')
        # inventaire final relu depuis les zips
        inv = []
        for p_zip in sorted((OUT / 'S2_parts').glob('*.zip')):
            with zipfile.ZipFile(p_zip) as z:
                for n in z.namelist():
                    inv.append(dict(cle=os.path.splitext(n)[0], fichier=n, part=p_zip.name))
        inv = pd.DataFrame(inv)
        attendu = set(ech.cle) - set(manquantes_total)
        assert set(inv.cle) == attendu, 'inventaire des zips ≠ échantillon attendu'
        ecrire_df(inv, 'S2_inventaire_images')
        marque('S2', f'{len(inv)} images dans {inv.part.nunique()} archives')
    except Exception as e:
        echec('S2', e)
else:
    print('S2 : déjà fait')

inv = lire_df('S2_inventaire_images'); inv['cle'] = inv['cle'].astype(str)
ech = ech[ech.cle.isin(set(inv.cle))].reset_index(drop=True)
# mise en scène locale (lecture rapide) : copie des zips puis décompression
if not (LOCAL / 'images' / '.ok').exists() or len(list((LOCAL / 'images').glob('*'))) < len(inv):
    pr = Progres(len(inv.part.unique()), 'copie locale des archives')
    for pz in sorted(inv.part.unique()):
        with zipfile.ZipFile(OUT / 'S2_parts' / pz) as z:
            z.extractall(LOCAL / 'images')
        pr()
    pr.fin()
    (LOCAL / 'images' / '.ok').write_text('ok')
FICHIER = dict(zip(inv.cle, inv.fichier))
print(f'{len(ech)} images disponibles localement')


## S3 : features des huit encodeurs gelés

Même recette qu analyses2 : redimensionnement direct à la résolution de l encodeur, ToTensor sans normalisation, RGB, bf16 ; moyenne des tokens de patch, token de classe exclu. Un fichier `features/S3_<encodeur>.npz` par encodeur, écrit puis relu avant d être accepté. La lecture des images se fait par lot depuis le disque local, donc la mémoire ne dépend pas du nombre d images.

In [ ]:
# ============================================================================
# S3 — FEATURES DES ENCODEURS GELÉS (moyenne des tokens de patch), un fichier par encodeur
# Recette identique à analyses2 : Resize((px,px)) + ToTensor, RGB, bf16 ; token de classe exclu.
# ============================================================================
import torchvision.transforms as TT

def charge_enc(nm):
    importlib.invalidate_caches()
    if nm in ('IN1K', 'DINOv2'):
        import timm
        spec = ('vit_base_patch16_224.augreg_in1k', 256) if nm == 'IN1K' else ('vit_base_patch14_dinov2.lvd142m', 224)
        m = timm.create_model(spec[0], pretrained=True, img_size=spec[1], num_classes=0)
        m._cls = True
        return m.to(DEVICE).eval(), spec[1]
    from src.models.vision_transformer import vit_base
    e = vit_base(patch_size=16, img_size=[256]); e._cls = False
    if nm.startswith('RANDOM'):
        torch.manual_seed(0 if nm == 'RANDOM' else 1)
        return e.to(DEVICE).eval(), 256
    ck = torch.load(ENCODEURS[nm], map_location='cpu', weights_only=False)
    e.load_state_dict({k.replace('module.', ''): v for k, v in ck['target_encoder'].items()}, strict=False)
    del ck; gc.collect()
    return e.to(DEVICE).eval(), 256

def tag_enc(nm):
    return nm.replace(' ', '_')

def verifie_images_local(cles):
    """Vérifie que chaque image se décode ; retourne (clés lisibles, clés illisibles). Rien n est gardé en mémoire."""
    ok, ko = [], []
    pr = Progres(len(cles), 'vérification images', chaque=200)
    for c in cles:
        try:
            with Image.open(LOCAL / 'images' / FICHIER[c]) as im:
                im.load()
            ok.append(c)
        except Exception:
            ko.append(c)
        pr()
    pr.fin()
    return ok, ko

def charge_lot(cles, tf):
    xs = []
    for c in cles:
        with Image.open(LOCAL / 'images' / FICHIER[c]) as im:
            xs.append(tf(im.convert('RGB')))
    return torch.stack(xs)

def features(enc, px, cles):
    """Lecture par lot depuis le disque local : la mémoire vive ne dépend pas du nombre d images."""
    tf = TT.Compose([TT.Resize((px, px)), TT.ToTensor()])
    Z = []
    pr = Progres(len(cles), f'features {px}px', chaque=BS_FEAT)
    with torch.inference_mode():
        for i in range(0, len(cles), BS_FEAT):
            x = charge_lot(cles[i:i + BS_FEAT], tf).to(DEVICE, non_blocking=True)
            ctx = torch.autocast('cuda', dtype=torch.bfloat16) if DEVICE == 'cuda' else torch.autocast('cpu', enabled=False)
            with ctx:
                z = enc.forward_features(x) if hasattr(enc, 'forward_features') else enc(x)
            g = px // (14 if px == 224 else 16)
            if getattr(enc, '_cls', False) and z.shape[1] == g * g + 1:
                z = z[:, 1:]
            Z.append(z.float().mean(1).cpu()); pr(len(x))
    pr.fin()
    return torch.cat(Z).numpy().astype(np.float32)

def lire_features(nm):
    p = OUT / 'features' / f'S3_{tag_enc(nm)}.npz'
    if not p.exists():
        return None
    d = np.load(p, allow_pickle=False)
    return pd.DataFrame(d['X'], index=d['cles'].astype(str))

if a_faire('S3'):
    try:
        cles = list(ech.cle)
        cles_ok, cles_ko = verifie_images_local(cles)
        if cles_ko:
            ecrire_texte(OUT / 'S3_illisibles.csv', 'cle\n' + '\n'.join(cles_ko))
            jrn(f'S3 : {len(cles_ko)} images illisibles retirées (S3_illisibles.csv)')
        assert len(cles_ko) / len(cles) < 0.01, 'trop d images illisibles'
        for nm in ENCODEURS:
            p = OUT / 'features' / f'S3_{tag_enc(nm)}.npz'
            prev = lire_features(nm)
            if prev is not None and set(prev.index) == set(cles_ok):
                print(f'  {nm} : features déjà présentes ({len(prev)})'); continue
            t0 = time.time(); jrn(f'S3 : {nm} : extraction')
            enc, px = charge_enc(nm)
            X = features(enc, px, cles_ok)
            del enc; gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            assert X.shape == (len(cles_ok), 768) and np.isfinite(X).all(), f'{nm} : features invalides {X.shape}'
            def _sauve(t, X=X, cles_ok=cles_ok):
                with open(t, 'wb') as fh:          # descripteur ouvert : numpy n ajoute pas d extension
                    np.savez(fh, X=X, cles=np.array(cles_ok))
            ecrire_atomique(p, _sauve)
            relu = lire_features(nm)
            assert relu is not None and relu.shape == (len(cles_ok), 768), f'{nm} : relecture échouée'
            jrn(f'S3 : {nm} : {X.shape} en {time.time() - t0:.0f}s, écrit et relu')
        gc.collect()
        marque('S3', f'{len(ENCODEURS)} encodeurs × {len(cles_ok)} images')
    except Exception as e:
        echec('S3', e)
else:
    print('S3 : déjà fait')

FEAT = {nm: lire_features(nm) for nm in ENCODEURS}
communes = set.intersection(*[set(f.index) for f in FEAT.values()])
ech = ech[ech.cle.isin(communes)].reset_index(drop=True)
for nm in FEAT:
    FEAT[nm] = FEAT[nm].loc[ech.cle].to_numpy(dtype=np.float32)
print({nm: FEAT[nm].shape for nm in FEAT})


## S4 : sondes anatomie

Régression logistique multinomiale sur features standardisées (statistiques du train), deux pondérations (équilibrée, brute), deux conditions : découpage par image (sources partagées) et sources tenues hors (prédictions hors pli regroupées). Intervalles bootstrap par image et par grappe de sources, test de permutation à l intérieur des plis, lignes de base (classe majoritaire ; majorité de la source pour le découpage par image), matrices de confusion. Sorties `metrics/S4_*`.

In [ ]:
# ============================================================================
# S4 — SONDES ANATOMIE : découpage par image (sources partagées) et sources tenues hors
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

Y_ANA = ech.anatomie.to_numpy()
SRC   = ech.source.to_numpy()
GRP   = ech.groupe.to_numpy()
PLI   = ech.pli.to_numpy()
IMG_TR = (ech.split_image == 'train').to_numpy()
CAT_INDEX = {c: i for i, c in enumerate(CATEGORIES)}

def ajuste_predit(Xtr, ytr, Xte, equilibre):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    clf = LogisticRegression(max_iter=3000, C=1.0, class_weight='balanced' if equilibre else None)
    clf.fit((Xtr - mu) / sd, ytr)
    P = clf.predict_proba((Xte - mu) / sd)
    P_full = np.zeros((len(Xte), len(CATEGORIES)), np.float32)
    for j, c in enumerate(clf.classes_):
        P_full[:, CAT_INDEX[c]] = P[:, j]
    return np.array(CATEGORIES)[P_full.argmax(1)], P_full

def metriques(y, yhat):
    labs = [c for c in CATEGORIES if (y == c).any()]
    rec = {c: float(((yhat == c) & (y == c)).sum() / (y == c).sum()) for c in labs}
    return dict(exactitude=float((y == yhat).mean()),
                exactitude_equilibree=float(np.mean([rec[c] for c in labs])),
                f1_macro=float(f1_score(y, yhat, labels=labs, average='macro', zero_division=0)),
                **{f'rappel_{c}': rec.get(c, np.nan) for c in CATEGORIES})

def predictions(nm, condition, equilibre):
    """Retourne un DataFrame de prédictions : une ligne par image de test (hors pli si condition == 'source')."""
    X = FEAT[nm]; lignes = []
    if condition == 'image':
        tr, te = np.where(IMG_TR)[0], np.where(~IMG_TR)[0]
        yhat, P = ajuste_predit(X[tr], Y_ANA[tr], X[te], equilibre)
        plis_te = np.full(len(te), -1)
    else:
        te_all, yhat_all, P_all, plis_te = [], [], [], []
        for k in range(K):
            tr, te = np.where(PLI != k)[0], np.where(PLI == k)[0]
            yh, P = ajuste_predit(X[tr], Y_ANA[tr], X[te], equilibre)
            te_all.append(te); yhat_all.append(yh); P_all.append(P); plis_te.append(np.full(len(te), k))
        te, yhat, P, plis_te = map(np.concatenate, (te_all, yhat_all, P_all, plis_te))
    df = pd.DataFrame(dict(cle=ech.cle.values[te], source=SRC[te], groupe=GRP[te], anatomie=Y_ANA[te], encodeur=nm,
                           condition=condition, variante='equilibree' if equilibre else 'brute',
                           pli=plis_te, y_pred=yhat))
    for j, c in enumerate(CATEGORIES):
        df[f'p_{c}'] = P[:, j]
    return df

def bootstrap(df, B, rng):
    """IC à 95 % par image et par grappe (groupe de sources apparentées) sur les prédictions regroupées."""
    y, yh, s = df.anatomie.to_numpy(), df.y_pred.to_numpy(), df.groupe.to_numpy()
    out = []
    obs = metriques(y, yh)
    # par image
    stats = defaultdict(list)
    for _ in range(B):
        i = rng.integers(0, len(y), len(y))
        m = metriques(y[i], yh[i])
        for k_ in ('exactitude', 'exactitude_equilibree', 'f1_macro'):
            stats[k_].append(m[k_])
    for k_, v in stats.items():
        out.append(dict(type='image', statistique=k_, estimation=obs[k_],
                        ic_bas=float(np.percentile(v, 2.5)), ic_haut=float(np.percentile(v, 97.5))))
    # par grappe de sources
    srcs = np.unique(s); idx_par_src = {u: np.where(s == u)[0] for u in srcs}
    stats = defaultdict(list)
    for _ in range(B):
        pick = rng.choice(srcs, len(srcs), replace=True)
        i = np.concatenate([idx_par_src[u] for u in pick])
        m = metriques(y[i], yh[i])
        for k_ in ('exactitude', 'exactitude_equilibree', 'f1_macro'):
            stats[k_].append(m[k_])
    for k_, v in stats.items():
        out.append(dict(type='grappe_sources', statistique=k_, estimation=obs[k_],
                        ic_bas=float(np.percentile(v, 2.5)), ic_haut=float(np.percentile(v, 97.5))))
    return out

def permutation(df, B, rng):
    """Niveau de hasard : étiquettes permutées à l intérieur de chaque pli (ou du test unique)."""
    y, yh, pl = df.anatomie.to_numpy(), df.y_pred.to_numpy(), df.pli.to_numpy()
    obs = metriques(y, yh)['exactitude_equilibree']
    nulls = []
    for _ in range(B):
        yp = y.copy()
        for k_ in np.unique(pl):
            i = np.where(pl == k_)[0]; yp[i] = y[rng.permutation(i)]
        nulls.append(metriques(yp, yh)['exactitude_equilibree'])
    nulls = np.array(nulls)
    return dict(observe=obs, perm_moyenne=float(nulls.mean()), perm_p975=float(np.percentile(nulls, 97.5)),
                p_valeur=float((1 + (nulls >= obs).sum()) / (1 + B)))

def lignes_metriques(df):
    """Métriques regroupées et par pli, sous forme de lignes de table."""
    base = dict(encodeur=df.encodeur.iloc[0], condition=df.condition.iloc[0], variante=df.variante.iloc[0])
    out = [dict(**base, pli='regroupe', n_test=len(df), **metriques(df.anatomie.to_numpy(), df.y_pred.to_numpy()))]
    if (df.pli >= 0).any():
        for k_, g in df.groupby('pli'):
            out.append(dict(**base, pli=str(k_), n_test=len(g), **metriques(g.anatomie.to_numpy(), g.y_pred.to_numpy())))
    return out

if a_faire('S4'):
    try:
        rng = np.random.default_rng(GRAINE)
        preds, mets, boots, perms, confs = [], [], [], [], []
        combos = [(nm, cond, eq) for nm in ENCODEURS for cond in ('image', 'source') for eq in (True, False)]
        pr = Progres(len(combos), 'sondes anatomie')
        for nm, cond, eq in combos:
            df = predictions(nm, cond, eq)
            preds.append(df); mets += lignes_metriques(df)
            base = dict(encodeur=nm, condition=cond, variante='equilibree' if eq else 'brute')
            for b in bootstrap(df, N_BOOT, rng):
                boots.append(dict(**base, **b))
            perms.append(dict(**base, **permutation(df, N_PERM, rng)))
            ct = pd.crosstab(df.anatomie, df.y_pred).reindex(index=CATEGORIES, columns=CATEGORIES, fill_value=0)
            for v in CATEGORIES:
                for p_ in CATEGORIES:
                    confs.append(dict(**base, vrai=v, predit=p_, n=int(ct.loc[v, p_])))
            jrn(f'S4 : {nm} | {cond} | {base["variante"]} : exactitude équilibrée '
                f'{metriques(df.anatomie.to_numpy(), df.y_pred.to_numpy())["exactitude_equilibree"]:.3f}')
            pr()
        pr.fin()
        # lignes de base
        bases = []
        tr, te = np.where(IMG_TR)[0], np.where(~IMG_TR)[0]
        maj = Counter(Y_ANA[tr]).most_common(1)[0][0]
        bases.append(dict(condition='image', baseline='classe_majoritaire', **metriques(Y_ANA[te], np.full(len(te), maj))))
        maj_src = {s: Counter(Y_ANA[tr][SRC[tr] == s]).most_common(1)[0][0] for s in np.unique(SRC[tr])}
        bases.append(dict(condition='image', baseline='majorite_de_la_source',
                          **metriques(Y_ANA[te], np.array([maj_src.get(s, maj) for s in SRC[te]]))))
        y_all, yh_all = [], []
        for k in range(K):
            tr, te = np.where(PLI != k)[0], np.where(PLI == k)[0]
            m_ = Counter(Y_ANA[tr]).most_common(1)[0][0]
            y_all.append(Y_ANA[te]); yh_all.append(np.full(len(te), m_))
        bases.append(dict(condition='source', baseline='classe_majoritaire', **metriques(np.concatenate(y_all), np.concatenate(yh_all))))
        ecrire_df(pd.concat(preds, ignore_index=True), 'S4_predictions')
        ecrire_df(pd.DataFrame(mets), 'S4_metriques')
        ecrire_df(pd.DataFrame(boots), 'S4_bootstrap')
        ecrire_df(pd.DataFrame(perms), 'S4_permutation')
        ecrire_df(pd.DataFrame(confs), 'S4_confusions')
        ecrire_df(pd.DataFrame(bases), 'S4_baselines')
        relu = lire_df('S4_metriques')
        assert relu is not None and len(relu[relu.pli == 'regroupe']) == len(combos), 'S4 : relecture incomplète'
        marque('S4', f'{len(combos)} sondes, {len(pd.concat(preds))} prédictions')
    except Exception as e:
        echec('S4', e)
else:
    print('S4 : déjà fait')

S4_MET = lire_df('S4_metriques'); S4_MET['pli'] = S4_MET['pli'].astype(str)
S4_BOOT, S4_PERM, S4_BASE, S4_CONF = lire_df('S4_bootstrap'), lire_df('S4_permutation'), lire_df('S4_baselines'), lire_df('S4_confusions')
print(S4_MET[(S4_MET.pli == 'regroupe') & (S4_MET.variante == 'equilibree')]
      .pivot(index='encodeur', columns='condition', values='exactitude_equilibree').round(3))


## S5 : contrôles d identification de la source

Sonde entraînée à reconnaître la source sur des images tenues hors des mêmes sources, globalement puis à catégorie fixée (mesure de généralisation, pas d apprentissage). Sortie `metrics/S5_source`.

In [ ]:
# ============================================================================
# S5 — CONTRÔLES : identification de la source (globale, et à catégorie fixée)
# Découpage par image à l intérieur de chaque source : mesure la généralisation de la
# signature d acquisition à de NOUVELLES images de la même source (pas une mesure en apprentissage).
# ============================================================================
def sonde_source(X, y_src, masque_tr, masque_te):
    mu, sd = X[masque_tr].mean(0), X[masque_tr].std(0) + 1e-6
    clf = LogisticRegression(max_iter=3000, C=1.0)
    clf.fit((X[masque_tr] - mu) / sd, y_src[masque_tr])
    yh = clf.predict((X[masque_te] - mu) / sd)
    y = y_src[masque_te]
    labs = np.unique(y)
    rec = [float(((yh == c) & (y == c)).sum() / (y == c).sum()) for c in labs]
    part_maj = float(max(Counter(y).values()) / len(y))
    return dict(n_sources=int(len(np.unique(y_src[masque_tr]))), n_test=int(len(y)),
                exactitude=float((y == yh).mean()), exactitude_equilibree=float(np.mean(rec)),
                hasard=float(1 / len(np.unique(y_src[masque_tr]))), majorite=part_maj)

if a_faire('S5'):
    try:
        lignes = []
        pr = Progres(len(ENCODEURS) * (1 + len(CATEGORIES)), 'sondes source')
        for nm in ENCODEURS:
            X = FEAT[nm]
            lignes.append(dict(encodeur=nm, perimetre='globale', categorie='toutes',
                               **sonde_source(X, SRC, IMG_TR, ~IMG_TR)))
            pr()
            for c in CATEGORIES:
                m = (Y_ANA == c)
                lignes.append(dict(encodeur=nm, perimetre='par_categorie', categorie=c,
                                   **sonde_source(X, SRC, m & IMG_TR, m & ~IMG_TR)))
                pr()
            jrn(f'S5 : {nm} : source globale {lignes[-5]["exactitude"]:.3f} (hasard {lignes[-5]["hasard"]:.3f})')
        pr.fin()
        ecrire_df(pd.DataFrame(lignes), 'S5_source')
        assert len(lire_df('S5_source')) == len(lignes)
        marque('S5', f'{len(lignes)} sondes source')
    except Exception as e:
        echec('S5', e)
else:
    print('S5 : déjà fait')
S5_SRC = lire_df('S5_source')
print(S5_SRC.pivot(index='encodeur', columns='categorie', values='exactitude').round(3))


## S6 : ablation par centrage par source

La moyenne des features de chaque source est retirée (sans étiquette), puis la sonde « sources tenues hors » est refaite. Si la signature d acquisition est surtout un décalage moyen, l exactitude remonte. Désactivable par `FAIRE_S6 = False`.

In [ ]:
# ============================================================================
# S6 — ABLATION : centrage des features par source (moyenne de chaque source retirée, sans étiquette)
# Si la signature d acquisition est surtout un décalage moyen, la sonde « sources tenues hors » remonte.
# ============================================================================
if FAIRE_S6 and a_faire('S6'):
    try:
        rng = np.random.default_rng(GRAINE + 6)
        mets, boots, preds = [], [], []
        pr = Progres(len(ENCODEURS), 'centrage par source')
        for nm in ENCODEURS:
            X = FEAT[nm].copy()
            for s in np.unique(SRC):
                m = SRC == s
                X[m] -= X[m].mean(0, keepdims=True)
            FEAT_ORIG = FEAT[nm]; FEAT[nm] = X
            try:
                df = predictions(nm, 'source', True)
            finally:
                FEAT[nm] = FEAT_ORIG
            df['variante'] = 'equilibree_centree'
            preds.append(df); mets += lignes_metriques(df)
            for b in bootstrap(df, N_BOOT, rng):
                boots.append(dict(encodeur=nm, condition='source', variante='equilibree_centree', **b))
            jrn(f'S6 : {nm} : exactitude équilibrée centrée {metriques(df.anatomie.to_numpy(), df.y_pred.to_numpy())["exactitude_equilibree"]:.3f}')
            pr()
        pr.fin()
        ecrire_df(pd.concat(preds, ignore_index=True), 'S6_predictions')
        ecrire_df(pd.DataFrame(mets), 'S6_metriques'); ecrire_df(pd.DataFrame(boots), 'S6_bootstrap')
        assert len(lire_df('S6_metriques')) == len(mets)
        marque('S6', f'{len(ENCODEURS)} encodeurs')
    except Exception as e:
        echec('S6', e)
elif FAIRE_S6:
    print('S6 : déjà fait')
S6_MET = lire_df('S6_metriques') if FAIRE_S6 else None
S6_BOOT = lire_df('S6_bootstrap') if FAIRE_S6 else None
if S6_MET is not None:
    S6_MET['pli'] = S6_MET['pli'].astype(str)
    print(S6_MET[S6_MET.pli == 'regroupe'][['encodeur', 'exactitude_equilibree']].round(3).to_string(index=False))


## S7 : tables, figures, README, verdict, manifeste

Tables T1 à T6 (CSV, Markdown, LaTeX), figures F1 à F5 (PNG 300 dpi et SVG), `README_resultats.md` avec le paragraphe de méthodes et de résultats en anglais, `VERDICT.txt` selon la règle pré-enregistrée, `MANIFESTE.csv` (taille et SHA-256 de chaque fichier). Cette étape est toujours ré-exécutée ; les versions précédentes sont archivées.

In [ ]:
# ============================================================================
# S7 — TABLES, FIGURES, README, VERDICT, MANIFESTE (toujours ré-exécuté ; l ancien est archivé)
# ============================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ORDRE = [e for e in ['RANDOM', 'IN1K', 'DINOv2', 'A0 ep100', 'A0 ep200', 'A0 ep300', 'FOETAL ep50', 'FOETAL ep100'] if e in ENCODEURS]
ORDRE += [e for e in ENCODEURS if e not in ORDRE]
LABEL = {'RANDOM': 'Random init.', 'IN1K': 'ImageNet-1k', 'DINOv2': 'DINOv2', 'A0 ep100': 'OBSonix A0 ep100',
         'A0 ep200': 'OBSonix A0 ep200', 'A0 ep300': 'OBSonix A0 ep300', 'FOETAL ep50': 'OBSonix fetal ep50',
         'FOETAL ep100': 'OBSonix fetal ep100'}
CAT_EN = {'foetal': 'fetal', 'cardiaque': 'cardiac', 'thyroide': 'thyroid', 'sein': 'breast'}
SURFACE, INK, INK2, MUTED, GRID, AXIS = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
BLUE, ORANGE, AQUA, YELLOW = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8.5, 'axes.edgecolor': AXIS, 'axes.linewidth': 0.8,
                     'xtick.color': INK2, 'ytick.color': INK2, 'axes.labelcolor': INK2, 'text.color': INK,
                     'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE})

def tidy(ax, axe='y'):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.tick_params(length=0); ax.grid(True, axis=axe, color=GRID, linewidth=0.8); ax.set_axisbelow(True)

def sauve_fig(fig, nom):
    for ext in ('png', 'svg'):
        p = OUT / 'figures' / f'{nom}.{ext}'
        if p.exists():
            shutil.copy2(p, OUT / 'archive' / f'{nom}_{datetime.now():%Y%m%d_%H%M%S}.{ext}')
        ecrire_atomique(p, lambda t: fig.savefig(t, dpi=300, facecolor=SURFACE, format=ext))
    plt.close(fig); print(f'  figure : {nom}')

def sauve_table(df, nom, index=False):
    ecrire_atomique(OUT / 'tables' / f'{nom}.csv', lambda t: df.to_csv(t, index=index))
    try:
        md = df.to_markdown(index=index)
    except Exception:
        md = df.to_string(index=index)
    ecrire_texte(OUT / 'tables' / f'{nom}.md', md)
    try:
        ecrire_texte(OUT / 'tables' / f'{nom}.tex', df.to_latex(index=index, float_format='%.3f'))
    except Exception as e:
        jrn(f'{nom}.tex non écrit ({type(e).__name__})')
    print(f'  table : {nom}')

def ic(df_boot, nm, cond, var, typ, stat='exactitude_equilibree'):
    r = df_boot[(df_boot.encodeur == nm) & (df_boot.condition == cond) & (df_boot.variante == var)
                & (df_boot.type == typ) & (df_boot.statistique == stat)]
    return (float(r.estimation.iloc[0]), float(r.ic_bas.iloc[0]), float(r.ic_haut.iloc[0])) if len(r) else (np.nan, np.nan, np.nan)

def fmt_ic(t):
    return f'{t[0]:.3f} [{t[1]:.3f}; {t[2]:.3f}]'

try:
    reg = S4_MET[S4_MET.pli == 'regroupe']
    # ---------------------------------------------------------------- T1
    lignes = []
    for var in ('equilibree', 'brute'):
        for nm in ORDRE:
            r_img = reg[(reg.encodeur == nm) & (reg.condition == 'image') & (reg.variante == var)].iloc[0]
            r_src = reg[(reg.encodeur == nm) & (reg.condition == 'source') & (reg.variante == var)].iloc[0]
            plis = S4_MET[(S4_MET.encodeur == nm) & (S4_MET.condition == 'source') & (S4_MET.variante == var) & (S4_MET.pli != 'regroupe')]
            perm = S4_PERM[(S4_PERM.encodeur == nm) & (S4_PERM.condition == 'source') & (S4_PERM.variante == var)].iloc[0]
            lignes.append({'Probe weighting': var, 'Encoder': LABEL.get(nm, nm),
                           'Image-wise balanced accuracy [95% CI, image bootstrap]': fmt_ic(ic(S4_BOOT, nm, 'image', var, 'image')),
                           'Source-held-out balanced accuracy [95% CI, source-cluster bootstrap]': fmt_ic(ic(S4_BOOT, nm, 'source', var, 'grappe_sources')),
                           'Source-held-out, mean ± SD across folds': f'{plis.exactitude_equilibree.mean():.3f} ± {plis.exactitude_equilibree.std(ddof=0):.3f}',
                           'Image-wise accuracy': round(float(r_img.exactitude), 3),
                           'Source-held-out accuracy': round(float(r_src.exactitude), 3),
                           'Source-held-out macro-F1': round(float(r_src.f1_macro), 3),
                           'Permutation null, 97.5th percentile': round(float(perm.perm_p975), 3),
                           'Permutation p value': round(float(perm.p_valeur), 4)})
    for b in S4_BASE.itertuples(index=False):
        lignes.append({'Probe weighting': 'baseline', 'Encoder': f'{b.baseline} ({b.condition})',
                       'Image-wise balanced accuracy [95% CI, image bootstrap]': f'{b.exactitude_equilibree:.3f}' if b.condition == 'image' else '',
                       'Source-held-out balanced accuracy [95% CI, source-cluster bootstrap]': f'{b.exactitude_equilibree:.3f}' if b.condition == 'source' else '',
                       'Source-held-out, mean ± SD across folds': '',
                       'Image-wise accuracy': round(float(b.exactitude), 3) if b.condition == 'image' else np.nan,
                       'Source-held-out accuracy': round(float(b.exactitude), 3) if b.condition == 'source' else np.nan,
                       'Source-held-out macro-F1': np.nan, 'Permutation null, 97.5th percentile': np.nan, 'Permutation p value': np.nan})
    T1 = pd.DataFrame(lignes); sauve_table(T1, 'T1_anatomy_probe_main')

    # ---------------------------------------------------------------- T2 rappel par catégorie (sources tenues hors, équilibrée)
    T2 = pd.DataFrame([{'Encoder': LABEL.get(nm, nm),
                        **{f'Recall {CAT_EN[c]}': round(float(reg[(reg.encodeur == nm) & (reg.condition == 'source') & (reg.variante == 'equilibree')][f'rappel_{c}'].iloc[0]), 3) for c in CATEGORIES}}
                       for nm in ORDRE]); sauve_table(T2, 'T2_recall_by_category_source_held_out')

    # ---------------------------------------------------------------- T3 plan des plis
    eff = ech.pivot_table(index='pli', columns='anatomie', values='cle', aggfunc='count', fill_value=0)
    T3 = pd.DataFrame([{'Fold': int(k), 'Held-out sources (family in brackets when grouped)': '; '.join(f'{r.source} [{r.groupe}]' if r.groupe != r.source else r.source for r in plan[plan.pli == k].itertuples()),
                        **{f'{CAT_EN[c]} images': int(eff.loc[k, c]) if c in eff.columns else 0 for c in CATEGORIES},
                        'Total images': int(eff.loc[k].sum())} for k in sorted(eff.index)])
    sauve_table(T3, 'T3_fold_design')

    # ---------------------------------------------------------------- T4 source
    T4 = pd.DataFrame([{'Encoder': LABEL.get(nm, nm),
                        'All sources: accuracy': round(float(S5_SRC[(S5_SRC.encodeur == nm) & (S5_SRC.perimetre == 'globale')].exactitude.iloc[0]), 3),
                        'All sources: chance': round(float(S5_SRC[(S5_SRC.encodeur == nm) & (S5_SRC.perimetre == 'globale')].hasard.iloc[0]), 3),
                        **{f'Within {CAT_EN[c]}: accuracy (chance)': (f'{float(S5_SRC[(S5_SRC.encodeur == nm) & (S5_SRC.categorie == c)].exactitude.iloc[0]):.3f} '
                                                                    f'({float(S5_SRC[(S5_SRC.encodeur == nm) & (S5_SRC.categorie == c)].hasard.iloc[0]):.3f})') for c in CATEGORIES}}
                       for nm in ORDRE]); sauve_table(T4, 'T4_source_identification_control')

    # ---------------------------------------------------------------- T5 centrage
    EMBOITE = bool(ech.groupby('source').anatomie.nunique().max() == 1)   # chaque source ne contient qu une catégorie
    if S6_MET is not None:
        reg6 = S6_MET[S6_MET.pli == 'regroupe']
        T5 = pd.DataFrame([{'Encoder': LABEL.get(nm, nm),
                            'Source-held-out balanced accuracy, raw features [95% CI]': fmt_ic(ic(S4_BOOT, nm, 'source', 'equilibree', 'grappe_sources')),
                            'Source-held-out balanced accuracy, per-source centered [95% CI]': fmt_ic(ic(S6_BOOT, nm, 'source', 'equilibree_centree', 'grappe_sources')),
                            'Difference': round(float(reg6[reg6.encodeur == nm].exactitude_equilibree.iloc[0] - reg[(reg.encodeur == nm) & (reg.condition == 'source') & (reg.variante == 'equilibree')].exactitude_equilibree.iloc[0]), 3)}
                           for nm in ORDRE]); sauve_table(T5, 'T5_per_source_centering_ablation')

    # ---------------------------------------------------------------- T6 verdict pré-enregistré
    maj_src_bal = float(S4_BASE[(S4_BASE.condition == 'source') & (S4_BASE.baseline == 'classe_majoritaire')].exactitude_equilibree.iloc[0])
    verd = []
    for nm in ORDRE:
        est, lo, hi = ic(S4_BOOT, nm, 'source', 'equilibree', 'grappe_sources')
        p975 = float(S4_PERM[(S4_PERM.encodeur == nm) & (S4_PERM.condition == 'source') & (S4_PERM.variante == 'equilibree')].perm_p975.iloc[0])
        verd.append({'Encoder': LABEL.get(nm, nm), 'Balanced accuracy': round(est, 3), 'CI lower (source cluster)': round(lo, 3),
                     'Permutation 97.5th pct': round(p975, 3), 'Majority baseline': round(maj_src_bal, 3),
                     'Anatomy survives source hold-out': bool(lo > p975 and est > maj_src_bal)})
    T6 = pd.DataFrame(verd); sauve_table(T6, 'T6_preregistered_verdict')

    # ---------------------------------------------------------------- F1 exactitude équilibrée par découpage
    x = np.arange(len(ORDRE)); w = 0.36
    fig, ax = plt.subplots(figsize=(max(7.5, 0.95 * len(ORDRE) + 2), 4.0), dpi=300)
    for j, (cond, col, typ, lab) in enumerate((('image', BLUE, 'image', 'Image-wise split (sources shared)'),
                                               ('source', ORANGE, 'grappe_sources', 'Source-held-out split'))):
        vals = [ic(S4_BOOT, nm, cond, 'equilibree', typ) for nm in ORDRE]
        est = np.array([v[0] for v in vals]); lo = np.array([v[1] for v in vals]); hi = np.array([v[2] for v in vals])
        pos = x + (j - 0.5) * (w + 0.04)
        ax.bar(pos, est, width=w, color=col, edgecolor='none', label=lab)
        ax.errorbar(pos, est, yerr=[np.clip(est - lo, 0, None), np.clip(hi - est, 0, None)], fmt='none', ecolor=INK2, elinewidth=0.9, capsize=2)
        if cond == 'source':
            for p_, e_ in zip(pos, est):
                if e_ > 0.15:
                    ax.text(p_, e_ - 0.03, f'{e_:.2f}', ha='center', va='top', fontsize=6.8, color='#ffffff')
                else:
                    ax.text(p_ + w * 0.6, e_, f'{e_:.2f}', ha='left', va='center', fontsize=6.8, color=INK)
    ax.axhline(1 / len(CATEGORIES), color=MUTED, linewidth=0.9, linestyle=(0, (3, 2)))
    ax.text(x[-1] + 0.55, 1 / len(CATEGORIES) + 0.01, f'chance (1/{len(CATEGORIES)})', ha='right', va='bottom', fontsize=6.8, color=MUTED)
    ax.set_xticks(x); ax.set_xticklabels([LABEL.get(nm, nm) for nm in ORDRE], rotation=20, ha='right')
    ax.set_ylim(0, 1.12); ax.set_yticks([0, .25, .5, .75, 1]); ax.set_ylabel('Balanced accuracy (4 anatomical categories)')
    tidy(ax); ax.legend(loc='upper left', frameon=False, fontsize=7.5, ncol=2)
    ax.set_title(f'Anatomy linear probe: image-wise versus source-held-out evaluation\n'
                 f'({len(ech):,} images, {ech.source.nunique()} sources, {K} folds; error bars: 95% bootstrap CI)', loc='left', fontsize=9)
    fig.tight_layout(); sauve_fig(fig, 'F1_balanced_accuracy_by_split')

    # ---------------------------------------------------------------- F2 matrices de confusion
    trio = list(dict.fromkeys(e for e in ('RANDOM', TEMOIN, PRINCIPAL) if e in ENCODEURS))
    fig, axes = plt.subplots(1, len(trio), figsize=(3.2 * len(trio) + 0.5, 3.4), dpi=300)
    axes = np.atleast_1d(axes)
    for ax, nm in zip(axes, trio):
        c = S4_CONF[(S4_CONF.encodeur == nm) & (S4_CONF.condition == 'source') & (S4_CONF.variante == 'equilibree')]
        M = c.pivot(index='vrai', columns='predit', values='n').reindex(index=CATEGORIES, columns=CATEGORIES).to_numpy(dtype=float)
        Mn = M / np.clip(M.sum(1, keepdims=True), 1, None)
        ax.imshow(Mn, cmap=matplotlib.colors.LinearSegmentedColormap.from_list('b', ['#f2f7fd', '#0d366b']), vmin=0, vmax=1)
        for i_ in range(len(CATEGORIES)):
            for j_ in range(len(CATEGORIES)):
                ax.text(j_, i_, f'{Mn[i_, j_]:.2f}', ha='center', va='center', fontsize=7,
                        color='#ffffff' if Mn[i_, j_] > 0.55 else INK)
        ax.set_xticks(range(len(CATEGORIES))); ax.set_xticklabels([CAT_EN[c_] for c_ in CATEGORIES], rotation=30, ha='right')
        ax.set_yticks(range(len(CATEGORIES))); ax.set_yticklabels([CAT_EN[c_] for c_ in CATEGORIES])
        ax.set_title(LABEL.get(nm, nm), fontsize=8.5, loc='left'); ax.tick_params(length=0)
        for s in ax.spines.values():
            s.set_visible(False)
    axes[0].set_ylabel('True category'); axes[len(trio) // 2].set_xlabel('Predicted category (row-normalized), source-held-out')
    fig.tight_layout(); sauve_fig(fig, 'F2_confusion_source_held_out')

    # ---------------------------------------------------------------- F3 dispersion par pli
    fig, ax = plt.subplots(figsize=(max(7.0, 0.9 * len(ORDRE) + 2), 3.6), dpi=300)
    for i_, nm in enumerate(ORDRE):
        plis = S4_MET[(S4_MET.encodeur == nm) & (S4_MET.condition == 'source') & (S4_MET.variante == 'equilibree') & (S4_MET.pli != 'regroupe')]
        ax.scatter(np.full(len(plis), i_) + np.linspace(-0.12, 0.12, len(plis)), plis.exactitude_equilibree, s=26, color=ORANGE, edgecolor=SURFACE, linewidth=1, zorder=3)
        est = ic(S4_BOOT, nm, 'source', 'equilibree', 'grappe_sources')[0]
        ax.plot([i_ - 0.25, i_ + 0.25], [est, est], color=INK, linewidth=1.4, zorder=4)
    ax.axhline(1 / len(CATEGORIES), color=MUTED, linewidth=0.9, linestyle=(0, (3, 2)))
    ax.set_xticks(range(len(ORDRE))); ax.set_xticklabels([LABEL.get(nm, nm) for nm in ORDRE], rotation=20, ha='right')
    ax.set_ylim(0, 1.05); ax.set_ylabel('Balanced accuracy, source-held-out'); tidy(ax)
    ax.set_title('Per-fold balanced accuracy (dots) and pooled out-of-fold estimate (bar)', loc='left', fontsize=9)
    fig.tight_layout(); sauve_fig(fig, 'F3_per_fold_dispersion')

    # ---------------------------------------------------------------- F4 identification de la source à catégorie fixée
    fig, ax = plt.subplots(figsize=(max(7.5, 0.95 * len(ORDRE) + 2), 4.0), dpi=300)
    cols = [BLUE, ORANGE, AQUA, YELLOW]; w = 0.19
    for j, c in enumerate(CATEGORIES):
        r = S5_SRC[(S5_SRC.perimetre == 'par_categorie') & (S5_SRC.categorie == c)].set_index('encodeur').reindex(ORDRE)
        pos = x + (j - 1.5) * (w + 0.02)
        ax.bar(pos, r.exactitude.values, width=w, color=cols[j], edgecolor='none', label=f'{CAT_EN[c]} ({int(r.n_sources.iloc[0])} sources)')
        ax.hlines(r.hasard.values, pos - w / 2, pos + w / 2, color=INK, linewidth=1.0)
    ax.plot([], [], color=INK, linewidth=1.0, label='chance (1 / number of sources)')
    ax.set_xticks(x); ax.set_xticklabels([LABEL.get(nm, nm) for nm in ORDRE], rotation=20, ha='right')
    ax.set_ylim(0, 1.15); ax.set_yticks([0, .25, .5, .75, 1]); ax.set_ylabel('Source identification accuracy (held-out images)')
    tidy(ax); ax.legend(loc='upper left', frameon=False, fontsize=7, ncol=3)
    ax.set_title('Acquisition-source identification within a fixed anatomical category', loc='left', fontsize=9)
    fig.tight_layout(); sauve_fig(fig, 'F4_source_identification_within_category')

    # ---------------------------------------------------------------- F5 centrage
    if S6_MET is not None:
        fig, ax = plt.subplots(figsize=(6.5, 0.42 * len(ORDRE) + 1.6), dpi=300)
        y = np.arange(len(ORDRE))[::-1]
        for yi, nm in zip(y, ORDRE):
            a = ic(S4_BOOT, nm, 'source', 'equilibree', 'grappe_sources')[0]
            b = ic(S6_BOOT, nm, 'source', 'equilibree_centree', 'grappe_sources')[0]
            ax.plot([a, b], [yi, yi], color=AXIS, linewidth=2, zorder=1)
            ax.plot(a, yi, 'o', color=ORANGE, markersize=7, markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=2)
            ax.plot(b, yi, 'o', color=AQUA, markersize=7, markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=2)
        ax.plot([], [], 'o', color=ORANGE, label='raw features'); ax.plot([], [], 'o', color=AQUA, label='per-source centered features')
        ax.axvline(1 / len(CATEGORIES), color=MUTED, linewidth=0.9, linestyle=(0, (3, 2)))
        ax.set_yticks(y); ax.set_yticklabels([LABEL.get(nm, nm) for nm in ORDRE]); ax.set_xlim(0, 1.0)
        ax.set_xlabel('Balanced accuracy, source-held-out'); tidy(ax, 'x')
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), frameon=False, fontsize=7.5, ncol=2)
        ax.set_title('Effect of removing each source\'s mean feature vector' + (' (not informative: one category per source)' if EMBOITE else ''), loc='left', fontsize=9)
        fig.tight_layout(); sauve_fig(fig, 'F5_per_source_centering')

    # ---------------------------------------------------------------- README et verdict
    def v(nm, cond, typ):
        return ic(S4_BOOT, nm, cond, 'equilibree', typ)
    g_src = S5_SRC[S5_SRC.perimetre == 'globale'].set_index('encodeur')
    surv = T6.set_index('Encoder')['Anatomy survives source hold-out']
    verdict = ('SURVIT' if bool(surv.get(LABEL.get(PRINCIPAL, PRINCIPAL), False)) else 'NE SURVIT PAS')
    n_src = ech.source.nunique()
    txt_en = (f"Methods. From the {255887 if not MOCK else len(pd.read_csv(IDX)):,}-image pretraining corpus index, we retained the four anatomical categories present in at least three source datasets "
              f"with at least {MIN_CELLULE} images per (category, source) cell ({', '.join(CAT_EN[c] for c in CATEGORIES)}), sampled at most {PLAFOND} images per cell "
              f"({len(ech):,} images from {n_src} sources), and assigned whole sources (datasets from the same institution or acquisition campaign grouped together: {', '.join(sorted(set(FAMILLES.values())))}) to {K} folds so that every fold held out at least one source of each category (Table T3). "
              f"Frozen encoders produced mean-pooled patch-token features ({CONFIG['pretraitement']}); a multinomial logistic-regression probe with class-balanced weights was trained on features standardized with training statistics. "
              f"Two designs were compared on the same images: an image-wise split (70/30 within each source, sources shared between training and test) and a source-held-out design, in which each image is predicted by a probe that never saw its source. "
              f"The primary outcome was balanced accuracy of pooled out-of-fold predictions, with 95% bootstrap confidence intervals by image and by source cluster ({N_BOOT} resamples) and a within-fold label-permutation null ({N_PERM} permutations). "
              f"As controls, a probe was trained to identify the acquisition source on held-out images of the same sources, overall and within each category, and the source-held-out probe was repeated after subtracting each source's mean feature vector.\n\n"
              f"Results. For {LABEL.get(PRINCIPAL, PRINCIPAL)}, balanced accuracy was {v(PRINCIPAL, 'image', 'image')[0]:.3f} [{v(PRINCIPAL, 'image', 'image')[1]:.3f}; {v(PRINCIPAL, 'image', 'image')[2]:.3f}] under the image-wise split and "
              f"{v(PRINCIPAL, 'source', 'grappe_sources')[0]:.3f} [{v(PRINCIPAL, 'source', 'grappe_sources')[1]:.3f}; {v(PRINCIPAL, 'source', 'grappe_sources')[2]:.3f}] with sources held out "
              f"(permutation null 97.5th percentile {float(S4_PERM[(S4_PERM.encodeur == PRINCIPAL) & (S4_PERM.condition == 'source') & (S4_PERM.variante == 'equilibree')].perm_p975.iloc[0]):.3f}; majority baseline {maj_src_bal:.3f}). "
              f"For {LABEL.get(TEMOIN, TEMOIN)}, the corresponding values were {v(TEMOIN, 'image', 'image')[0]:.3f} and {v(TEMOIN, 'source', 'grappe_sources')[0]:.3f} [{v(TEMOIN, 'source', 'grappe_sources')[1]:.3f}; {v(TEMOIN, 'source', 'grappe_sources')[2]:.3f}]; "
              f"for a randomly initialized encoder, {v('RANDOM', 'image', 'image')[0]:.3f} and {v('RANDOM', 'source', 'grappe_sources')[0]:.3f} [{v('RANDOM', 'source', 'grappe_sources')[1]:.3f}; {v('RANDOM', 'source', 'grappe_sources')[2]:.3f}]. "
              f"Acquisition source was identified among {int(g_src.loc[PRINCIPAL, 'n_sources'])} sources on held-out images with accuracy {float(g_src.loc[PRINCIPAL, 'exactitude']):.3f} for {LABEL.get(PRINCIPAL, PRINCIPAL)} "
              f"and {float(g_src.loc['RANDOM', 'exactitude']):.3f} for the random encoder (chance {float(g_src.loc[PRINCIPAL, 'hasard']):.3f}). Per-category recall, per-fold dispersion, within-category source identification and the centering ablation are given in Tables T2 to T5 and Figures F1 to F5. "
              + (f"Because every source in this sample contains a single anatomical category, per-source centering removes the category signal by construction (balanced accuracy {1 / len(CATEGORIES):.2f} for all encoders) and cannot separate acquisition signature from anatomy here. " if EMBOITE else "")
              + (f"A randomly initialized encoder also exceeded the permutation null with sources held out, so part of the source-held-out signal is available from low-level image statistics rather than learned representations." if bool(surv.get(LABEL.get('RANDOM', 'RANDOM'), False)) else ""))
    readme = f'''# Résultats de la campagne {CAMPAGNE}

Généré le {datetime.now():%Y-%m-%d %H:%M}. Données : `metrics/` (CSV et Parquet), tables publiables : `tables/` (CSV, Markdown, LaTeX), figures : `figures/` (PNG 300 dpi et SVG), journal : `JOURNAL.md`, pré-enregistrement : `S0_preenregistrement.md`, manifeste : `MANIFESTE.csv`.

## Verdict pré-enregistré
Pour l encodeur principal ({LABEL.get(PRINCIPAL, PRINCIPAL)}), le signal anatomique {verdict} à la tenue hors des sources
(critère : borne inférieure de l IC à 95 % par grappe de sources > 97,5e centile de permutation, et estimation > ligne de base majoritaire).
Détail par encodeur dans `tables/T6_preregistered_verdict.csv`.
{'Attention : l encodeur à initialisation aléatoire satisfait lui aussi la règle ; une partie du signal « sources tenues hors » vient donc de statistiques d image de bas niveau, pas des représentations apprises. La comparaison utile est celle entre encodeurs (IC par grappe de sources, larges avec ' + str(ech.groupe.nunique()) + ' groupes).' if bool(surv.get(LABEL.get('RANDOM', 'RANDOM'), False)) else ''}
{'Ablation par centrage (T5, F5) : non informative ici, chaque source ne contenant qu une catégorie, le centrage par source retire la catégorie par construction.' if EMBOITE else ''}

## Lecture rapide (exactitude équilibrée, sonde équilibrée)
{T1[T1['Probe weighting'] == 'equilibree'][['Encoder', 'Image-wise balanced accuracy [95% CI, image bootstrap]', 'Source-held-out balanced accuracy [95% CI, source-cluster bootstrap]']].to_string(index=False)}

## Paragraphe prêt pour le manuscrit (anglais)
{txt_en}

## Fichiers
- T1_anatomy_probe_main : table principale (les deux pondérations de sonde, lignes de base)
- T2_recall_by_category_source_held_out : rappel par catégorie, sources tenues hors
- T3_fold_design : plan des plis (sources tenues hors et effectifs)
- T4_source_identification_control : identification de la source, globale et à catégorie fixée
- T5_per_source_centering_ablation : effet du centrage par source (si activé ; dégénéré quand chaque source n a qu une catégorie)
- T6_preregistered_verdict : application de la règle de décision
- F1 à F5 : figures correspondantes
'''
    ecrire_texte(OUT / 'README_resultats.md', readme)
    ecrire_texte(OUT / 'VERDICT.txt', f'{verdict}\n{T6.to_string(index=False)}\n')
    # ---------------------------------------------------------------- manifeste
    man = [dict(fichier=str(p.relative_to(OUT)), octets=p.stat().st_size, sha256=sha256_fichier(p))
           for p in sorted(OUT.rglob('*')) if p.is_file() and 'archive' not in p.parts and p.name != 'MANIFESTE.csv']
    ecrire_atomique(OUT / 'MANIFESTE.csv', lambda t: pd.DataFrame(man).to_csv(t, index=False))
    marque('S7', f'{len(man)} fichiers dans le manifeste ; verdict {verdict}')
    print('\n' + T6.to_string(index=False)); print('\nVERDICT :', verdict)
except Exception as e:
    echec('S7', e)


## S8 : dépouillement de la lecture en aveugle des paires pHash

Lit `analyses2/adjudication/grille_de_lecture.csv` ; si toutes les paires ont une catégorie de lecteur 1, ouvre la clé et calcule le taux de positifs confirmés (catégories 1 à 3) par distance de Hamming, le kappa de Cohen entre lecteurs et les désaccords. Sinon, la clé n est pas ouverte et l étape est reportée. Sorties `tables/T7`, `tables/T8`.

In [ ]:
# ============================================================================
# S8 — DÉPOUILLEMENT DE LA LECTURE EN AVEUGLE DES PAIRES pHash (analyses2/adjudication)
# La clé n est ouverte QUE si la grille est complète (colonne lecteur1_categorie remplie pour toutes les paires).
# Rien n est modifié dans analyses2 : lecture seule, résultats écrits ici.
# ============================================================================
CATS_ADJ = {1: 'same file', 2: 'identical pixels', 3: 'same frame, transformed', 4: 'adjacent frames',
            5: 'similar but independent', 6: 'undeterminable'}

def kappa_cohen(a, b):
    a, b = np.asarray(a), np.asarray(b); labs = np.unique(np.concatenate([a, b]))
    po = float((a == b).mean())
    pe = float(sum(((a == l).mean()) * ((b == l).mean()) for l in labs))
    return (po - pe) / (1 - pe) if pe < 1 else np.nan

def lire_csv_souple(p, colonne_requise):
    """CSV lu quel que soit l enregistrement (Excel français « ; », BOM, CRLF, identifiant suivi de virgules) ;
    en-têtes en minuscules, identifiants ramenés à PAIRE0000."""
    brut = Path(p).read_bytes().decode('utf-8-sig', errors='replace')
    entete = brut.splitlines()[0] if brut.strip() else ''
    df = None
    for sep in sorted((';', ',', '\t'), key=lambda c: -entete.count(c)):
        try:
            d = pd.read_csv(io.StringIO(brut), sep=sep, dtype=str, keep_default_na=False, engine='python')
        except Exception:
            continue
        d.columns = [str(c).strip().lower().strip('\ufeff"') for c in d.columns]
        if colonne_requise in d.columns:
            df = d.loc[:, [c for c in d.columns if c and not c.startswith('unnamed')]]; break
    if df is None:
        raise KeyError(f'{Path(p).name} : colonne « {colonne_requise} » introuvable ; en-tête lu : {entete[:200]!r}')
    df = df.apply(lambda col: col.str.strip())
    if 'identifiant' in df.columns:
        ext = df['identifiant'].str.upper().str.extract(r'(PAIRE\s*0*\d+)', expand=False).str.replace(r'\s+', '', regex=True)
        ext = ext.str.replace(r'PAIRE0*(\d+)', lambda m: f'PAIRE{int(m.group(1)):04d}', regex=True)
        df['identifiant'] = ext.fillna(df['identifiant'].str.strip(' ,;\t"'))
    return df

if GRILLE_ADJ.exists():
    try:
        g = lire_csv_souple(GRILLE_ADJ, 'lecteur1_categorie')
    except Exception as e_:
        g = None
        print(f'S8 : grille illisible ({e_}) ; étape ignorée, rien n est ouvert'); jrn(f'S8 : grille illisible : {e_}')
    for c in ('lecteur2_categorie', 'arbitre', 'commentaire'):
        if g is not None and c not in g.columns:
            g[c] = ''
    complet = g is not None and len(g) > 0 and (g['lecteur1_categorie'].str.extract(r'([1-6])', expand=False).notna()).all()
    if g is None:
        pass
    elif not complet:
        n_faits = int(g['lecteur1_categorie'].str.extract(r'([1-6])', expand=False).notna().sum())
        print(f'S8 : grille de lecture incomplète ({n_faits}/{len(g)} paires lues) ; clé non ouverte, dépouillement reporté.')
        jrn(f'S8 : grille incomplète ({n_faits}/{len(g)}), reporté')
    elif a_faire('S8'):
        try:
            cle = lire_csv_souple(CLE_ADJ, 'hamming')
            df = g.merge(cle, on='identifiant', how='left')
            manquantes = df['hamming'].isna() | (df['hamming'] == '')
            assert not manquantes.any(), f'S8 : identifiants de la grille absents de la clé : {df.loc[manquantes, "identifiant"].tolist()}'
            for c in ('lecteur1_categorie', 'lecteur2_categorie', 'arbitre'):
                df[c] = pd.to_numeric(df[c].str.extract(r'([1-6])', expand=False), errors='coerce')
            df['hamming'] = pd.to_numeric(df['hamming'], errors='coerce')
            df['pixels_identiques'] = df['pixels_identiques'].astype(str).str.lower().eq('true')
            deux = df['lecteur2_categorie'].notna().all()
            accord = (~deux) | (df['lecteur1_categorie'] == df['lecteur2_categorie'])
            df['finale'] = np.where(df['arbitre'].notna(), df['arbitre'], np.where(accord, df['lecteur1_categorie'], np.nan))
            non_arbitres = int(df['finale'].isna().sum())
            if non_arbitres:
                jrn(f'S8 : {non_arbitres} désaccord(s) sans arbitrage : paires exclues du décompte final tant que la colonne arbitre est vide')
            df['positif_principal'] = df['finale'].isin([1, 2, 3])
            res = [dict(mesure='paires lues', valeur=len(df)),
                   dict(mesure='positifs principaux (catégories 1 à 3)', valeur=f'{int(df.positif_principal.sum())}/{len(df)} ({100 * df.positif_principal.mean():.1f} %)'),
                   dict(mesure='même clip, même trame ou trame adjacente (catégories 1 à 4)', valeur=f'{int(df.finale.isin([1, 2, 3, 4]).sum())}/{len(df)} ({100 * df.finale.isin([1, 2, 3, 4]).mean():.1f} %)'),
                   dict(mesure='catégorie 4 (trames adjacentes)', valeur=int((df.finale == 4).sum())),
                   dict(mesure='catégorie 5 (indépendantes)', valeur=int((df.finale == 5).sum())),
                   dict(mesure='catégorie 6 (indéterminable)', valeur=int((df.finale == 6).sum()))]
            if deux:
                res.append(dict(mesure='kappa de Cohen lecteur 1 vs lecteur 2', valeur=round(kappa_cohen(df.lecteur1_categorie, df.lecteur2_categorie), 3)))
                res.append(dict(mesure='désaccords entre lecteurs', valeur=int(((df.lecteur1_categorie != df.lecteur2_categorie)).sum())))
                res.append(dict(mesure='désaccords non arbitrés (exclus)', valeur=non_arbitres))
            par_d = df.groupby('hamming').agg(n=('identifiant', 'size'), positifs=('positif_principal', 'sum'),
                                              pixels_identiques=('pixels_identiques', 'sum')).reset_index()
            par_d['taux_positifs'] = (par_d.positifs / par_d.n).round(3)
            ecrire_df(df[['identifiant', 'hamming', 'pixels_identiques', 'lecteur1_categorie', 'lecteur2_categorie', 'arbitre', 'finale', 'positif_principal', 'commentaire']], 'S8_adjudication_detail')
            ecrire_df(pd.DataFrame(res), 'S8_adjudication_resume'); ecrire_df(par_d, 'S8_adjudication_par_distance')
            sauve_table(pd.DataFrame(res).rename(columns={'mesure': 'Measure', 'valeur': 'Value'}), 'T7_blinded_adjudication_summary')
            sauve_table(par_d.rename(columns={'hamming': 'Hamming distance', 'n': 'Pairs', 'positifs': 'Confirmed positives (cat. 1-3)',
                                              'pixels_identiques': 'Identical pixel arrays', 'taux_positifs': 'Confirmation rate'}), 'T8_blinded_adjudication_by_distance')
            marque('S8', f'{len(df)} paires dépouillées')
            print(pd.DataFrame(res).to_string(index=False))
        except Exception as e:
            echec('S8', e)
    else:
        print('S8 : déjà fait')
else:
    print(f'S8 : grille introuvable ({GRILLE_ADJ}) ; étape ignorée')


## S9 : publication des résultats dans le dépôt GitHub

Lit le jeton dans le secret Colab `GOLDBACH` (jamais affiché, jamais écrit), compare chaque fichier de la liste blanche (README, verdict, pré-enregistrement, journal, manifeste, tables, figures, métriques CSV) à la version du dépôt `Fetal-odyssey/ObSonix` et pousse en un seul commit les fichiers nouveaux ou modifiés dans `experiments/EXP25_ANATOMY_PROBE_SOURCE_HELD_OUT/`, avec un `README.md` (badge Colab, verdict, tables T6 et T2, figures), `summary.md` et `summary.json` au format des autres expériences, et une copie du notebook (sorties retirées) dans `notebooks/`. Rien du corpus, aucune feature, aucune archive. Si le secret est absent ou si GitHub est injoignable, l étape est ignorée et les résultats restent sur le Drive.

In [ ]:
# ============================================================================
# S9 — PUBLICATION DES RÉSULTATS DANS LE DÉPÔT GITHUB (secret Colab GOLDBACH)
# ============================================================================
# Toujours ré-exécutée : un seul commit par session, ne contenant que les fichiers nouveaux ou modifiés.
# Le jeton est lu dans le secret Colab, jamais affiché, jamais écrit dans un fichier ni dans une URL.
GH_DEPOT    = 'Fetal-odyssey/ObSonix'
GH_BRANCHE  = None                                              # None : branche par défaut du dépôt
GH_DOSSIER  = 'experiments/EXP25_ANATOMY_PROBE_SOURCE_HELD_OUT'  # dossier de destination des résultats
GH_NOTEBOOK = 'notebooks/OBSonix_sonde_anatomie4.ipynb'         # copie du notebook (sorties retirées)
GH_SECRETS  = ['GOLDBACH', 'OBSONIX3', 'GITHUB_TOKEN2', 'GITHUB_TOKEN']   # essayés dans l ordre ; le premier qui PEUT ÉCRIRE dans le dépôt est retenu
GH_SECRET   = GH_SECRETS[0]
GH_MAX_MO   = 40                                                # au-delà, le fichier reste sur le Drive et est listé dans le README
NB_DRIVE    = Path('/content/drive/MyDrive/Colab Notebooks/OBSonix_sonde_anatomie4.ipynb')
import base64, re, requests

GH_API = (os.environ.get('OBSONIX_MOCK_GITHUB_API') or 'https://api.github.com') if MOCK else 'https://api.github.com'
MOTIF_JETON = re.compile(r'(?:gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,})')

def lire_secret(nom, patient):
    """Valeur du secret Colab `nom`, ou None. patient : attend l accord d accès jusqu à 90 s (premier secret seulement)."""
    if MOCK:
        d = json.loads(os.environ.get('OBSONIX_MOCK_SECRETS') or '{}')
        if not d and os.environ.get('OBSONIX_MOCK_GITHUB_TOKEN'):
            d = {GH_SECRETS[0]: os.environ['OBSONIX_MOCK_GITHUB_TOKEN']}
        return (d.get(nom) or '').strip() or None
    try:
        from google.colab import userdata
    except ImportError:
        return None
    for tentative in range(3 if patient else 1):
        try:
            return (userdata.get(nom) or '').strip() or None
        except Exception as e:
            typ = type(e).__name__
            if typ == 'SecretNotFoundError':
                return None
            if typ == 'TimeoutException' and patient and tentative < 2:
                print(f'S9 : en attente de l accord d accès au secret {nom} : cliquer « Accorder l accès » dans la fenêtre Colab,\n'
                      f'     ou activer « Accès depuis le notebook » pour {nom} dans le panneau Secrets (icône clé) ; nouvel essai dans 30 s')
                time.sleep(30); continue
            print(f'S9 : secret {nom} inaccessible ({typ}) ; activer « Accès depuis le notebook » dans le panneau Secrets si ce secret doit servir')
            return None
    return None

def type_jeton(t):
    return 'à granularité fine' if t.startswith('github_pat_') else ('classique' if t.startswith(('ghp_', 'gho_')) else 'de type inconnu')

class GitHub:
    """Client minimal ; les messages d erreur ne contiennent jamais l en-tête d autorisation."""
    def __init__(self, jeton):
        self.s = requests.Session()
        self.s.headers.update({'Authorization': f'Bearer {jeton}', 'Accept': 'application/vnd.github+json',
                               'X-GitHub-Api-Version': '2022-11-28', 'User-Agent': 'OBSonix-notebook'})
    def __call__(self, methode, chemin, ok=(200, 201), essais=3, **kw):
        for i in range(essais):
            try:
                r = self.s.request(methode, GH_API + chemin, timeout=300, **kw)
            except requests.RequestException as e:
                if i == essais - 1:
                    raise RuntimeError(f'GitHub injoignable ({type(e).__name__}) sur {methode} {chemin}')
                time.sleep(5 * (i + 1)); continue
            if r.status_code in ok:
                return r.json() if r.content else {}
            if r.status_code >= 500 and i < essais - 1:
                time.sleep(5 * (i + 1)); continue
            try:
                msg = r.json().get('message', r.text[:300])
            except Exception:
                msg = r.text[:300]
            attendu = r.headers.get('X-Accepted-GitHub-Permissions')
            if attendu:
                msg += f' ; permissions attendues : {attendu}'
            raise RuntimeError(f'GitHub {r.status_code} sur {methode} {chemin} : {msg}')
    def peut_ecrire(self):
        """Sonde d écriture : un blob orphelin minuscule (aucun effet visible sur le dépôt). Renvoie (ok, message)."""
        try:
            self('POST', f'/repos/{GH_DEPOT}/git/blobs', json={'content': 'OBSonix write probe\n', 'encoding': 'utf-8'})
            return True, ''
        except RuntimeError as e:
            return False, str(e)

def sha_blob(octets):
    return hashlib.sha1(b'blob %d\0' % len(octets) + octets).hexdigest()

def fichiers_resultats():
    """Liste blanche stricte : (chemin dans le dépôt, octets). Rien du corpus, pas de features, pas d archives."""
    sel = [OUT / n for n in ('README_resultats.md', 'VERDICT.txt', 'MANIFESTE.csv', 'S0_config.json',
                             'S0_preenregistrement.md', 'S1_resume.md', 'JOURNAL.md')]
    sel += sorted(p for p in (OUT / 'tables').glob('*') if p.suffix in ('.csv', '.md', '.tex'))
    sel += sorted(p for p in (OUT / 'figures').glob('*') if p.suffix in ('.png', '.svg'))
    sel += sorted((OUT / 'metrics').glob('*.csv'))
    out, trop_gros = [], []
    for p in sel:
        if not p.is_file() or p.suffix == '.tmp':
            continue
        rel = p.relative_to(OUT).as_posix()
        assert not ({'S2_parts', 'features', 'archive'} & set(p.parts)), f'S9 : fichier hors liste blanche refusé : {rel}'
        assert p.suffix not in ('.zip', '.npz', '.tar', '.parquet', '.jpg', '.jpeg'), f'S9 : type refusé : {rel}'
        if p.stat().st_size > GH_MAX_MO * 1e6:
            trop_gros.append((rel, p.stat().st_size)); continue
        out.append((f'{GH_DOSSIER}/{rel}', p.read_bytes()))
    return out, trop_gros

def notebook_courant():
    """Octets du notebook (session Colab, sinon copie Drive, sinon MOCK), sorties et métadonnées d exécution retirées."""
    nb, origine = None, ''
    if MOCK:
        p = os.environ.get('OBSONIX_MOCK_NOTEBOOK', '')
        if p and Path(p).exists():
            nb, origine = json.loads(Path(p).read_text(encoding='utf-8')), 'mock'
    else:
        try:
            from google.colab import _message
            rep = _message.blocking_request('get_ipynb', request='', timeout_sec=60)
            if isinstance(rep, dict) and isinstance(rep.get('ipynb'), dict) and rep['ipynb'].get('cells'):
                nb, origine = rep['ipynb'], 'session Colab'
        except Exception as e:
            jrn(f'S9 : notebook de la session non récupéré ({type(e).__name__}) ; essai de la copie Drive')
        if nb is None and NB_DRIVE.exists():
            nb, origine = json.loads(NB_DRIVE.read_text(encoding='utf-8')), str(NB_DRIVE)
    if nb is None:
        return None, origine
    for c in nb.get('cells', []):
        if isinstance(c.get('source'), list):
            c['source'] = ''.join(c['source'])
        if c.get('cell_type') == 'code':
            c['outputs'], c['execution_count'] = [], None
        for k in ('executionInfo', 'outputId', 'execution', 'collapsed'):
            c.get('metadata', {}).pop(k, None)
    nb.setdefault('metadata', {}).pop('widgets', None)
    nb['nbformat'], nb['nbformat_minor'] = nb.get('nbformat', 4), nb.get('nbformat_minor', 0)
    return (json.dumps(nb, ensure_ascii=False, indent=1, sort_keys=True) + '\n').encode('utf-8'), origine

def lire_texte(p):
    return p.read_text(encoding='utf-8') if p.exists() else ''

def readme_et_resume(branche, trop_gros, origine_nb):
    """README.md du dossier (rendu par GitHub), summary.md et summary.json (convention du dépôt)."""
    lien = f'https://colab.research.google.com/github/{GH_DEPOT}/blob/{branche}/{GH_NOTEBOOK}'
    exp_id = Path(GH_DOSSIER).name
    verdict = lire_texte(OUT / 'VERDICT.txt').splitlines()
    verdict_mot = verdict[0].strip() if verdict else 'indisponible'
    t6 = pd.read_csv(OUT / 'tables' / 'T6_preregistered_verdict.csv') if (OUT / 'tables' / 'T6_preregistered_verdict.csv').exists() else None
    t1 = pd.read_csv(OUT / 'tables' / 'T1_anatomy_probe_main.csv') if (OUT / 'tables' / 'T1_anatomy_probe_main.csv').exists() else None
    marq = lire_texte(ETATS / 'S7.ok').splitlines()
    date_s7 = marq[0] if marq else f'{datetime.now():%Y-%m-%d %H:%M:%S}'
    figures = sorted(p.name for p in (OUT / 'figures').glob('*.png'))
    tables = sorted(p.stem for p in (OUT / 'tables').glob('*.csv'))
    lab_p, lab_t = LABEL.get(PRINCIPAL, PRINCIPAL), LABEL.get(TEMOIN, TEMOIN)
    def ligne(df, enc):
        r = df[df.Encoder == enc]
        return r.iloc[0] if len(r) else None
    metr = {'principal': lab_p, 'temoin': lab_t, 'verdict_principal': verdict_mot,
            'n_images': int(len(ech)), 'n_sources': int(ech.source.nunique()), 'n_groupes_sources': int(ech.groupe.nunique()),
            'k_plis': int(K), 'categories': [CAT_EN[c] for c in CATEGORIES], 'campagne_drive': str(OUT)}
    if t6 is not None:
        for nm, lab in (('principal', lab_p), ('temoin', lab_t), ('random', LABEL.get('RANDOM', 'RANDOM'))):
            r = ligne(t6, lab)
            if r is not None:
                metr[f'{nm}_balanced_accuracy_source_held_out'] = float(r['Balanced accuracy'])
                metr[f'{nm}_ci_lower_source_cluster'] = float(r['CI lower (source cluster)'])
                metr[f'{nm}_permutation_p975'] = float(r['Permutation 97.5th pct'])
                metr[f'{nm}_survives'] = bool(r['Anatomy survives source hold-out'])
        metr['majority_baseline_source_held_out'] = float(t6['Majority baseline'].iloc[0])
    if t1 is not None:
        for nm, lab in (('principal', lab_p), ('temoin', lab_t)):
            r = t1[(t1['Probe weighting'] == 'equilibree') & (t1.Encoder == lab)]
            if len(r):
                metr[f'{nm}_balanced_accuracy_image_wise'] = r['Image-wise balanced accuracy [95% CI, image bootstrap]'].iloc[0]
    notes = (f'Sonde anatomie (fetal, cardiac, thyroid, breast) sur features gelees, comparaison decoupage par image contre sources tenues hors '
             f'({K} plis, familles de jeux tenues hors ensemble). Critere principal : exactitude equilibree, IC bootstrap par grappe de sources, '
             f'test de permutation, regle pre-enregistree. Verdict {lab_p} : le signal anatomique {verdict_mot} a la tenue hors des sources. '
             f'Images et features restent sur le Drive.')
    resume = dict(exp_id=exp_id, timestamp=date_s7[:16], metrics=metr, notes=notes)
    md_t6 = lire_texte(OUT / 'tables' / 'T6_preregistered_verdict.md')
    md_t2 = lire_texte(OUT / 'tables' / 'T2_recall_by_category_source_held_out.md')
    gros = ''.join(f'- `{rel}` ({n / 1e6:.0f} Mo, au-delà de {GH_MAX_MO} Mo)\n' for rel, n in trop_gros)
    readme = f'''# {exp_id} : sonde anatomie à quatre catégories, sources tenues hors

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)]({lien})

Campagne `{CAMPAGNE}` (Drive : `{OUT}`), publiée automatiquement par l’étape S9 du notebook [`{GH_NOTEBOOK}`](../../{GH_NOTEBOOK}) (résultats S7 du {date_s7}). Relancer le notebook depuis Colab avec le lien ci-dessus : chaque étape reprend là où elle s’est arrêtée et ce dossier est mis à jour à la fin.

## Question
La représentation gelée des encodeurs (aléatoire, ImageNet-1k, DINOv2, OBSonix A0 ep100/200/300, OBSonix fœtal ep50/100) sépare-t-elle l’anatomie ({', '.join(CAT_EN[c] for c in CATEGORIES)}) lorsque les sources d’acquisition du test n’ont jamais été vues par la sonde ? Catégories restreintes à celles présentes dans au moins trois sources avec au moins {MIN_CELLULE} images par cellule, {PLAFOND} images au plus par cellule, {len(ech):,} images de {ech.source.nunique()} sources ({ech.groupe.nunique()} groupes tenus hors), {K} plis, chaque pli de test contenant les quatre catégories.

## Verdict pré-enregistré
Pour l’encodeur principal ({lab_p}), le signal anatomique **{verdict_mot}** à la tenue hors des sources (critère : borne inférieure de l’IC à 95 % par grappe de sources > 97,5e centile de permutation, et estimation > ligne de base majoritaire).

{md_t6}

## Rappel par catégorie, sources tenues hors (sonde équilibrée)
{md_t2}

## Figures
{''.join(f'![{f[:-4]}](figures/{f})' + chr(10) + chr(10) for f in figures)}
## Contenu du dossier
- `README_resultats.md` : rapport complet, avec le paragraphe de méthodes et de résultats en anglais prêt pour le manuscrit
- `S0_preenregistrement.md`, `S0_config.json` : question, critère principal, règle de décision et paramètres, écrits avant les résultats
- `S1_resume.md` : plan d’échantillonnage, sources par pli
- `tables/` : {', '.join(tables)} (CSV, Markdown, LaTeX)
- `figures/` : {', '.join(f[:-4] for f in figures)} (PNG 300 dpi et SVG)
- `metrics/` : échantillon, prédictions hors pli, bootstrap, permutations, matrices de confusion, contrôles source, ablation par centrage
- `JOURNAL.md` : journal horodaté de toutes les sessions ; `MANIFESTE.csv` : taille et SHA-256 de chaque fichier du dossier Drive
- `summary.md`, `summary.json` : résumé au format des autres expériences du dépôt

## Non inclus dans le dépôt
Les images du corpus (`S2_parts/`), les features des encodeurs (`features/*.npz`), les versions Parquet et les archives des versions précédentes restent sur le Drive (`{OUT}`).
{gros}
Notebook publié depuis : {origine_nb or 'non publié à cette session'}.
'''
    lignes = '\n'.join(f'| {k} | {", ".join(map(str, v)) if isinstance(v, list) else v} |' for k, v in metr.items())
    summary_md = f'# {exp_id}\n*{resume["timestamp"]}*\n\n## Metriques\n| Metrique | Valeur |\n|---|---|\n{lignes}\n\n## Notes\n{notes}\n'
    return readme, summary_md, json.dumps(resume, ensure_ascii=False, indent=2) + '\n'

def publier(gh, fichiers, message):
    """Un seul commit : blobs, arbre sur base_tree, commit, mise à jour de la référence, puis vérification.
    Retourne (branche, sha du commit ou None si rien n a changé, nombre de fichiers poussés)."""
    depot = gh('GET', f'/repos/{GH_DEPOT}')
    assert depot.get('full_name', '').lower() == GH_DEPOT.lower(), f'S9 : dépôt inattendu {depot.get("full_name")}'
    branche = GH_BRANCHE or depot['default_branch']
    if not depot.get('permissions', {}).get('push', False):
        raise RuntimeError(f'le jeton {GH_SECRET} n a pas le droit d écrire dans {GH_DEPOT} (permissions.push absent)')
    for tentative in range(2):
        tete = gh('GET', f'/repos/{GH_DEPOT}/git/ref/heads/{branche}')['object']['sha']
        arbre_base = gh('GET', f'/repos/{GH_DEPOT}/git/commits/{tete}')['tree']['sha']
        arbre = gh('GET', f'/repos/{GH_DEPOT}/git/trees/{arbre_base}', params={'recursive': '1'})
        if arbre.get('truncated'):
            jrn('S9 : arbre distant tronqué par l API ; des fichiers inchangés pourront être renvoyés (sans conséquence)')
        distant = {t['path']: t['sha'] for t in arbre['tree'] if t['type'] == 'blob'}
        dossiers = {t['path'] for t in arbre['tree'] if t['type'] == 'tree'}
        num = re.match(r'experiments/(EXP\d+)', GH_DOSSIER)
        if num and GH_DOSSIER not in dossiers:
            homonymes = [d for d in dossiers if d.startswith(f'experiments/{num.group(1)}_') or d == f'experiments/{num.group(1)}']
            assert not homonymes, f'S9 : le numéro {num.group(1)} est déjà utilisé ({homonymes}) ; changer GH_DOSSIER'
        changes = [(c, o) for c, o in fichiers if distant.get(c) != sha_blob(o)]
        if not changes:
            return branche, None, 0
        pr = Progres(len(changes), 'envoi vers GitHub')
        entrees = []
        for chemin, octets in changes:
            b = gh('POST', f'/repos/{GH_DEPOT}/git/blobs',
                   json={'content': base64.b64encode(octets).decode('ascii'), 'encoding': 'base64'})
            assert b['sha'] == sha_blob(octets), f'S9 : empreinte différente après envoi de {chemin}'
            entrees.append({'path': chemin, 'mode': '100644', 'type': 'blob', 'sha': b['sha']}); pr()
        pr.fin()
        nouvel_arbre = gh('POST', f'/repos/{GH_DEPOT}/git/trees', json={'base_tree': arbre_base, 'tree': entrees})
        commit = gh('POST', f'/repos/{GH_DEPOT}/git/commits', json={'message': message, 'tree': nouvel_arbre['sha'], 'parents': [tete]})
        try:
            gh('PATCH', f'/repos/{GH_DEPOT}/git/refs/heads/{branche}', json={'sha': commit['sha'], 'force': False})
        except RuntimeError as e:
            if tentative == 0 and ('fast forward' in str(e).lower() or ' 422 ' in str(e) or ' 409 ' in str(e)):
                jrn('S9 : la branche a avancé pendant l envoi ; nouvelle tentative sur la tête actuelle'); continue
            raise
        # vérification indépendante : le commit est sur la branche et le README publié a la bonne empreinte
        assert gh('GET', f'/repos/{GH_DEPOT}/git/ref/heads/{branche}')['object']['sha'] == commit['sha'], 'S9 : la branche ne pointe pas sur le commit'
        gh('GET', f'/repos/{GH_DEPOT}/commits/{commit["sha"]}')
        attendu = {c: sha_blob(o) for c, o in fichiers}
        for chemin in (f'{GH_DOSSIER}/README.md', GH_NOTEBOOK):
            if chemin in attendu:
                c = gh('GET', f'/repos/{GH_DEPOT}/contents/{chemin}', params={'ref': branche})
                assert c.get('sha') == attendu[chemin], f'S9 : {chemin} publié différent de celui envoyé'
        return branche, commit['sha'], len(changes)
    raise RuntimeError('S9 : deux tentatives de mise à jour de la branche ont échoué')

AVEC_RESULTATS = fait('S7')
if not AVEC_RESULTATS:
    print('S9 : S7 non fait ; seule la copie du notebook sera publiée, les résultats le seront après S7')
try:
    jeton, gh, GH_SECRET = None, None, GH_SECRETS[0]
    for i_, nom_ in enumerate(GH_SECRETS):
        t_ = lire_secret(nom_, patient=(i_ == 0))
        if not t_:
            continue
        gh_ = GitHub(t_)
        try:
            qui_ = gh_('GET', '/user').get('login', '?')
        except RuntimeError as e_:
            print(f'S9 : secret {nom_} : jeton refusé par GitHub ({e_}) ; secret suivant'); continue
        ok_, msg_ = gh_.peut_ecrire()
        if ok_:
            jeton, gh, GH_SECRET, qui = t_, gh_, nom_, qui_
            jrn(f'S9 : secret {nom_} accepté (compte {qui}, jeton {type_jeton(t_)}) ; écriture vérifiée sur {GH_DEPOT}'); break
        print(f'S9 : secret {nom_} (compte {qui_}, jeton {type_jeton(t_)}) : lecture seule sur {GH_DEPOT} ({msg_}) ; secret suivant')
    if jeton is None:
        jrn(f'S9 : publication GitHub ignorée : aucun des secrets {GH_SECRETS} ne peut écrire dans {GH_DEPOT} ; les résultats restent sur le Drive')
        print(f'''
S9 : aucun jeton utilisable. Le dépôt {GH_DEPOT} appartient à l ORGANISATION {GH_DEPOT.split("/")[0]} : un jeton à granularité
     fine créé sous votre compte personnel ne la couvre pas, quelles que soient ses permissions (« Resource not accessible »).
     Deux solutions, au choix :
       1. jeton classique : github.com/settings/tokens, « Generate new token (classic) », portée « repo » ;
       2. jeton à granularité fine : github.com/settings/personal-access-tokens, Resource owner = {GH_DEPOT.split("/")[0]},
          Repository access = Only select repositories → {GH_DEPOT.split("/")[1]}, Permissions → Contents : Read and write
          (l organisation doit autoriser ces jetons : Organization settings → Third-party access → Personal access tokens).
     Enregistrer le jeton dans le secret Colab {GH_SECRETS[0]} (ou un autre nom de GH_SECRETS), activer « Accès depuis le
     notebook », puis relancer cette seule cellule.''')
    else:
        depot = gh('GET', f'/repos/{GH_DEPOT}')
        branche = GH_BRANCHE or depot['default_branch']
        jrn(f'S9 : dépôt {depot.get("full_name")} ; branche {branche}')
        fichiers, trop_gros = fichiers_resultats() if AVEC_RESULTATS else ([], [])
        nb_octets, origine_nb = notebook_courant()
        if AVEC_RESULTATS:
            readme, summary_md, summary_json = readme_et_resume(branche, trop_gros, origine_nb)
            fichiers += [(f'{GH_DOSSIER}/README.md', readme.encode('utf-8')),
                         (f'{GH_DOSSIER}/summary.md', summary_md.encode('utf-8')),
                         (f'{GH_DOSSIER}/summary.json', summary_json.encode('utf-8'))]
        if nb_octets is not None:
            fichiers.append((GH_NOTEBOOK, nb_octets))
        else:
            jrn('S9 : notebook introuvable (ni session Colab, ni copie Drive) ; seuls les résultats sont publiés')
        if not fichiers:
            raise RuntimeError('rien à publier (ni résultats, ni notebook)')
        # GARDE-FOUS : périmètre des chemins, absence de tout jeton dans les fichiers texte
        for chemin, octets in fichiers:
            assert chemin.startswith(GH_DOSSIER + '/') or chemin == GH_NOTEBOOK, f'S9 : chemin hors périmètre : {chemin}'
            assert '..' not in chemin and not chemin.startswith('/'), f'S9 : chemin invalide : {chemin}'
            if chemin.endswith(('.md', '.txt', '.csv', '.json', '.tex', '.svg', '.ipynb')):
                texte = octets.decode('utf-8', 'ignore')
                assert jeton not in texte and not MOTIF_JETON.search(texte), f'S9 : {chemin} ressemble à contenir un jeton ; publication refusée'
        total = sum(len(o) for _, o in fichiers)
        print(f'S9 : {len(fichiers)} fichiers candidats ({total / 1e6:.1f} Mo)' + (f' ; {len(trop_gros)} trop gros, laissés sur le Drive' if trop_gros else ''))
        if AVEC_RESULTATS:
            verdict_mot = lire_texte(OUT / 'VERDICT.txt').splitlines()[0].strip() if (OUT / 'VERDICT.txt').exists() else 'indisponible'
            message = (f'OBSonix anatomie4 : résultats {CAMPAGNE} ; verdict {LABEL.get(PRINCIPAL, PRINCIPAL)} : le signal anatomique '
                       f'{verdict_mot} à la tenue hors des sources\n\nPublié par l étape S9 du notebook {Path(GH_NOTEBOOK).name} '
                       f'({len(ech)} images, {ech.source.nunique()} sources, {K} plis).')
        else:
            message = f'OBSonix anatomie4 : copie du notebook {Path(GH_NOTEBOOK).name} (résultats de {CAMPAGNE} à venir)'
        branche, sha, n = publier(gh, fichiers, message)
        lien_colab = f'https://colab.research.google.com/github/{GH_DEPOT}/blob/{branche}/{GH_NOTEBOOK}'
        lien_dossier = f'https://github.com/{GH_DEPOT}/tree/{branche}/{GH_DOSSIER}'
        if sha is None:
            jrn(f'S9 : dépôt déjà à jour, aucun commit ({len(fichiers)} fichiers identiques)')
        else:
            jrn(f'S9 : commit {sha[:10]} sur {branche}, {n} fichiers envoyés : {lien_dossier}')
        if AVEC_RESULTATS:
            marque('S9', f'{"commit " + sha[:10] if sha else "déjà à jour"} ; {lien_dossier}')
            print(f'\nDossier des résultats : {lien_dossier}')
        print(f'Ouvrir le notebook dans Colab depuis GitHub : {lien_colab}')
except Exception as e:
    jrn(f'S9 : ÉCHEC {type(e).__name__} : {e}')
    with JR.open('a', encoding='utf-8') as f:
        f.write('```\n' + traceback.format_exc() + '```\n')
    print('\nS9 : la publication GitHub a échoué ; les résultats restent complets sur le Drive. Corriger puis relancer cette cellule.')


## S10 : état final

In [ ]:
# ============================================================================
# S10 — ÉTAT FINAL
# ============================================================================
print('=' * 72); print(f'CAMPAGNE {CAMPAGNE} : état des étapes'); print('=' * 72)
OPTIONNELLES = {'S6': 'optionnelle', 'S8': 'en attente de la lecture en aveugle', 'S9': 'publication GitHub ignorée ou échouée'}
for etape in ('S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9'):
    p = ETATS / f'{etape}.ok'
    if p.exists():
        lignes = p.read_text(encoding='utf-8').splitlines()
        print(f'  {etape} : fait le {lignes[0]} ; {lignes[1] if len(lignes) > 1 else ""}')
    else:
        print(f'  {etape} : non fait' + (f' ({OPTIONNELLES[etape]})' if etape in OPTIONNELLES else ''))
print('\nSorties :', OUT)
for sous in ('tables', 'figures'):
    fichiers = sorted((OUT / sous).glob('*'))
    print(f'  {sous}/ : {len(fichiers)} fichiers')
    for f in fichiers[:40]:
        print(f'     {f.name}')
v_ = OUT / 'VERDICT.txt'
if v_.exists():
    print('\n' + v_.read_text(encoding='utf-8'))
if fait('S9'):
    print(f'Dépôt GitHub : https://github.com/{GH_DEPOT}/tree/{GH_BRANCHE or "main"}/{GH_DOSSIER}')
    print(f'Relancer ce notebook depuis GitHub : https://colab.research.google.com/github/{GH_DEPOT}/blob/{GH_BRANCHE or "main"}/{GH_NOTEBOOK}')
jrn('=== session terminée ===')
